# LungCT — Thực nghiệm phân loại ảnh CT phổi bằng CNN và ViT

**Dự án KLTN2627 · Bản quy trình 1.5 · Ngày 24/10/2026**

## Mục tiêu nghiên cứu
Chúng tôi sử dụng notebook này để tổ chức lại quy trình chạy mô hình trong tài liệu đầu vào, huấn luyện và đánh giá bốn mô hình phân loại ảnh CT phổi: CNN, ViT Head, CNN–ViT và Adaptive AACNN–ViT. Mỗi ảnh được gán vào một trong ba lớp: **Benign — lành tính**, **Malignant — ác tính**, **Normal — bình thường**. Đây là bài toán phân loại từng ảnh, chưa phải chẩn đoán một bệnh nhân từ toàn bộ hồ sơ hay khối CT ba chiều.

Notebook được xây dựng từ `Demo_Test_Combined.ipynb` và `demo_test_combined.py`. [Liên kết notebook đầu vào](https://colab.research.google.com/drive/1EE1KoUm4LhRoNOHKCc2_hiw0Icr7KBq2?usp=sharing) được lưu để truy nguyên. Danh tính tác giả bài báo, repository gốc và giấy phép cần được đối chiếu riêng. Chúng tôi không xem kiến trúc có sẵn là kiến trúc mới do nhóm đề xuất.

Đây là **quy trình đã chỉnh sửa**, không phải bản tái lập nguyên trạng: có thay đổi về chia tập, cache, loss theo mẫu, cách chọn checkpoint và augmentation fusion. Những thay đổi được ghi ở bảng phía dưới. Mục tiêu là làm thí nghiệm có thể kiểm tra và lưu đủ đầu ra để tích hợp về sau; chưa có kết quả thực nghiệm để khẳng định chất lượng được cải thiện.

## Cách chạy từ đầu đến cuối
1. Mở notebook bằng Google Colab và chọn runtime có GPU.
2. Sửa `BASE_DIR` và `OUTPUT_ROOT` ở mục 2 cho đúng vị trí dữ liệu và thư mục lưu kết quả trên Drive. Thư mục dữ liệu phải chứa ba thư mục `Benign`, `Malignant`, `Normal`.
3. Chọn **Thời gian chạy → Chạy tất cả** và cấp quyền Drive khi Colab yêu cầu. Lần đầu cần Internet để tải pretrained weights.
4. Mặc định `SMOKE_TEST=False`: chạy thực nghiệm đầy đủ. `RUN_TEST_ONCE=True`: đánh giá một mô hình đã xác định trước trên tập test sau huấn luyện. Không cần đổi cờ giữa chừng.
5. Kết quả được lưu trong một thư mục run riêng. Nếu lỗi, đọc thông báo tại cell dừng; các cell sau chưa được coi là đã hoàn thành. Không suy diễn thời gian chạy khi chưa đo trên runtime thực.

Có thể đặt `SMOKE_TEST=True` để kiểm tra luồng bằng dữ liệu nhỏ; khi đó test chính thức tự bỏ qua và mọi kết quả đều có cờ smoke. Không sử dụng số liệu smoke trong báo cáo kết quả nghiên cứu. Mỗi lần chạy lại toàn bộ tạo một run mới, không ghi đè run trước.

## Sơ đồ đọc quy trình
Dữ liệu ảnh → kiểm tra và loại trùng → chia train/validation/test → trích CLS ViT → tạo batch → xây mô hình → train và chọn checkpoint bằng validation → lưu/nạp lại → test một lần → dự đoán ảnh → Grad-CAM → lưu manifest.

## Thuật ngữ cần nắm
| Thuật ngữ | Ý nghĩa trong notebook |
|---|---|
| Tensor | Mảng số nhiều chiều mà mô hình xử lý; ảnh RGB có chiều cao, chiều rộng và ba kênh màu |
| CNN | Mạng convolution học đặc trưng cục bộ từ ảnh |
| ViT | Vision Transformer; ở đây dùng pretrained ViT để trích vector đặc trưng |
| Pretrained | Trọng số đã được huấn luyện trước; không đồng nghĩa mô hình đã được huấn luyện trên dữ liệu của nghiên cứu này |
| CLS | Vector 768 số lấy từ token đầu ra đặc biệt của ViT để biểu diễn ảnh |
| Head | Các lớp phân loại nhận đặc trưng và tạo scores ba lớp |
| Epoch và batch | Một epoch đi qua tập train; một batch là nhóm mẫu xử lý trong một bước |
| Checkpoint | Trọng số được lưu tại một thời điểm train; chọn checkpoint bằng validation |
| Manifest | Tệp mô tả có cấu trúc để biết ảnh, mô hình và cấu hình đã dùng |
| Hash SHA256 | Dấu vân tay dữ liệu giúp kiểm tra trùng và nhận biết tệp bị thay đổi |
| Cache | Kết quả tính trước được lưu để tránh tính lại; chỉ được dùng khi đúng ảnh, thứ tự và cấu hình |
| Inference | Dùng mô hình đã train để dự đoán ảnh, không cập nhật trọng số |
| Grad-CAM | Bản đồ mức đóng góp của vùng đặc trưng CNN vào score mục tiêu, không phải đường viền u đã xác nhận |

## Thay đổi so với đầu vào
| Điểm trong bản đầu vào | Cách xử lý và ảnh hưởng |
|---|---|
| Mount Drive và cấu hình lặp | Gom về một vị trí để tránh hai cấu hình khác nhau |
| Baseline tiếp tục train ở cell vẽ | Mỗi mô hình train một lần; biểu đồ đọc history đã lưu |
| Cache chỉ đối chiếu số ảnh | Đối chiếu hash ảnh đúng thứ tự, processor, revision và môi trường |
| Chưa kiểm tra trùng trước split | Loại trùng byte/pixel trong manifest; chặn trùng khác nhãn; không xóa ảnh nguồn |
| Train/validation 70/30 | Cố định 70/15/15 sau làm sạch; test riêng không dùng chọn checkpoint |
| Ảnh CNN augmentation nhưng CLS ViT từ ảnh gốc | Mặc định tắt augmentation online fusion và MixUp, bổ sung augmentation offline đồng bộ hai nhánh; đây là thay đổi protocol |
| Loss reduce về một scalar cả batch | Trả loss theo từng mẫu để class weight gắn với đúng ảnh; khác train semantics nguồn |
| Chọn checkpoint A/B riêng | Giữ checkpoint tốt nhất toàn hai stage bằng validation macro F1 |
| Lambda và custom layer thiếu cấu hình | Dùng layer có serialization và thêm kiểm tra lưu/nạp |
| Chưa có Grad-CAM | Thêm Grad-CAM nhánh CNN, giữ CLS ViT cố định |
| Tiêu đề No-Adaptive nhưng chưa có builder | Chỉ huấn luyện bốn mô hình có implementation; chưa có ablation No-Adaptive |

Ngân sách adaptive dài hơn baseline và nguồn pretrained khác nhau; không kết luận so sánh kiến trúc công bằng nếu chưa thiết kế thí nghiệm tương ứng. Chia theo ảnh chưa chứng minh độc lập bệnh nhân. Notebook chưa chuyển mô hình sang Android/iOS.


## Mô hình chính và phạm vi xuất bản
Adaptive AACNN–ViT là mô hình chính của đợt thực nghiệm. Bản mặc định không giảm số mẫu hoặc epoch vì mục đích demo. Ngân sách train giữ 15 epoch mỗi baseline và 10 + 20 epoch Adaptive như cấu hình đầu vào; early stopping có thể dừng trước giới hạn nếu validation không cải thiện. Đây là chọn checkpoint theo thực nghiệm, không phải bỏ bớt pipeline. Không có cơ sở để tăng epoch tùy ý sẽ tốt hơn.

Gói tải về phải bao gồm CNN/fusion Keras **và** ViT pretrained/processor, class order, tiền xử lý, checkpoint và manifest. Đây là nguồn mô hình chính để chuyển đổi, chưa phải tệp cắm trực tiếp vào React Native. Chuyển đổi cho Android/iOS và Grad-CAM ngoại tuyến vẫn là bước tiếp theo cần kiểm chứng.


**Quyết định cấu trúc mô hình:** giữ nguyên Adaptive làm mô hình chính. Không thêm student hoặc thay backbone để làm nhẹ. Mục 21 tạo ứng viên lưu ViT bằng FP16 và đo sai lệch trên validation; chưa xác nhận giảm RAM/latency điện thoại.


## 1 Môi trường và nguồn gốc
Colab thường có sẵn TensorFlow và PyTorch. Chỉ cài gói còn thiếu, không tự nâng cấp toàn bộ môi trường. Ghi phiên bản thực tế; nếu lỗi tương thích thì dừng và giữ log, không âm thầm đổi kiến trúc. Môi trường khóa đầy đủ được lưu sau run, không cam kết tương thích mọi phiên bản.

### Mục đích và đầu ra
Chúng tôi kiểm tra các thư viện cần thiết, kết nối Drive và ghi thông tin runtime. TensorFlow thực hiện CNN/fusion, PyTorch thực hiện ViT. Hai framework có thể cùng sử dụng GPU nên cần quản lý bộ nhớ.

**Đầu vào:** runtime Colab và quyền đọc/ghi Drive. **Đầu ra:** các thư viện được import, Drive được mount; phiên bản đầy đủ sẽ lưu ở mục 2. Cell cài đặt chỉ thêm gói thiếu, không khóa cứng môi trường tác giả chưa được xác minh. Nếu runtime không có TensorFlow/PyTorch hoặc cài đặt lỗi, cần xử lý môi trường trước khi train.

In [ ]:
import importlib.util, subprocess, sys
for package in ("transformers", "huggingface_hub", "sklearn", "PIL", "ipywidgets"):
    if importlib.util.find_spec(package) is None:
        pip_name = {"sklearn": "scikit-learn", "PIL": "Pillow"}.get(package, package)
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

In [ ]:
import os
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
import gc, csv, json, math, time, random, hashlib, platform, subprocess
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import tensorflow as tf
# Giới hạn TF cấp phát dần để giảm tranh chấp VRAM với PyTorch.
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as error:
        print("Chưa đổi được memory growth:", error)
import torch, transformers
from transformers import ViTImageProcessor, ViTModel
from huggingface_hub import model_info
from tensorflow.keras import layers, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             accuracy_score, roc_curve, auc)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Chạy ngoài Colab: dùng đường dẫn cục bộ ở cấu hình.")

## 2 Cấu hình duy nhất
`CLASS_NAMES` là thứ tự đầu ra, không được đảo sau train. CNN nhận RGB 512×512 trong [0,1]; ResNet chuyển ngược về [0,255] rồi dùng preprocessing ImageNet. ViT xử lý bản RGB resize 224×224 bằng processor của checkpoint. Kích thước hiện được giữ như nguồn, chưa tối ưu điện thoại.

`SPLIT_MODE="holdout"` cố định tỷ lệ 70% train, 15% validation và 15% test, khác cách chia 70/30 trong tệp đầu vào. Chia sau làm sạch, theo seed và thứ tự đã lưu; không tái sử dụng số mẫu hoặc split cũ. Test không được dùng chọn checkpoint. Không chạy lại nhiều lần để chọn kết quả test đẹp nhất.

### Cách hiểu các tham số
- `BASE_DIR`: thư mục chứa dữ liệu; `OUTPUT_ROOT`: nơi ghi kết quả bền vững. Không đặt output bên trong thư mục lớp ảnh.
- `CLASS_NAMES`: thứ tự nhãn, cũng là thứ tự ba scores. Chỉ số 0 là Benign, 1 là Malignant, 2 là Normal trong cấu hình này.
- `IMG_SIZE`: kích thước CNN; `VIT_SIZE`: kích thước trước processor ViT. Đây là hai nhánh khác nhau, không phải ảnh tùy ý đổi kích thước mà vẫn giữ cùng thí nghiệm.
- `BATCH_SIZE`: số ảnh mỗi batch. Batch lớn tốn VRAM hơn; nếu thay để tránh hết bộ nhớ, ghi thay đổi vào run.
- `EPOCHS_BASELINE`, `EPOCHS_STAGE_A/B`: giới hạn epoch; early stopping có thể dừng sớm.
- `SEED`: hạt khởi tạo ngẫu nhiên để giảm biến thiên, không bảo đảm bitwise identical trên mọi GPU/môi trường.

**Đầu ra:** `config.json`, `environment.json`, `pip-freeze.txt` và thư mục run duy nhất. Một run là một lần thực nghiệm có cấu hình và đầu ra riêng. Kiểm tra đường dẫn in ra trước khi chạy các bước tốn thời gian.

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/KLTN2627/Dataset/IQ_OTHNCCD")
OUTPUT_ROOT = Path("/content/drive/MyDrive/KLTN2627/ket_qua_thuc_nghiem")
CACHE_ROOT = OUTPUT_ROOT / "vit_cache_v2"
CLASS_NAMES = ["Benign", "Malignant", "Normal"]
SEED = 42
IMG_SIZE = (512, 512)
VIT_SIZE = (224, 224)
BATCH_SIZE = 8
VIT_BATCH_SIZE = 8
VIT_MODEL_ID = "google/vit-base-patch16-224-in21k"
VIT_REVISION = "main"  # Được resolve thành commit cố định trước trích feature.
SPLIT_MODE = "holdout"  # Cố định 70% train / 15% validation / 15% test.
SMOKE_TEST = False
MODELS_TO_RUN = ["CNN", "ViT_Head", "CNN_ViT", "AACNN_Adaptive"]
EPOCHS_BASELINE = 15
EPOCHS_STAGE_A, EPOCHS_STAGE_B = 10, 20  # Đủ ngân sách nguồn; early stopping vẫn chọn best validation.
UNFREEZE_LAST_N = 60
FREEZE_BATCHNORM = True  # Thay đổi train policy: ổn định thống kê với batch nhỏ.
CNN_AUGMENT = False  # Dùng augmentation offline đồng bộ cho cả CNN và ViT.
FUSION_AUGMENT = False  # ViT cache từ ảnh gốc, tránh lệch hai nhánh ở mặc định.
SYNC_AUGMENT_COPIES = 1  # Một biến thể nhẹ mỗi ảnh train, ngoài ảnh gốc.
LIGHT_MAX_MACRO_F1_DROP = 0.01  # Ngưỡng kỹ thuật đề xuất, chỉ đánh giá trên validation.
LIGHT_MIN_LABEL_AGREEMENT = 0.98
MIXUP_ALPHA = 0.0  # Không triển khai MixUp trong bản quy trình này.
RUN_TEST_ONCE = True  # Run chính thức đánh giá test một lần sau khi huấn luyện.
PREPROCESS_VERSION = "rgb-nearest-source-resize-v1"

random.seed(SEED); np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
try: tf.config.experimental.enable_op_determinism()
except AttributeError: pass
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = OUTPUT_ROOT / (RUN_ID + ("_SMOKE" if SMOKE_TEST else "_EXPERIMENT"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
NUM_CLASSES = len(CLASS_NAMES)
assert SPLIT_MODE == "holdout", "Nghiên cứu này cố định chia 70/15/15."
assert MIXUP_ALPHA == 0, "MixUp chưa triển khai; không thay cấu hình rồi giả định đã hoạt động."
print("Thư mục kết quả:", RUN_DIR)

In [ ]:
def write_json(path, value):
    """Ghi JSON có Unicode; dùng cho cấu hình và kết quả có thể đối chiếu."""
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False,
                                   default=str, allow_nan=False), encoding="utf-8")

def digest_json(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()

def sha256_file(path):
    """Hash theo chunk để không nạp cả checkpoint lớn vào RAM."""
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""): h.update(chunk)
    return h.hexdigest()

CONFIG_KEYS = ["CLASS_NAMES", "SEED", "IMG_SIZE", "VIT_SIZE", "BATCH_SIZE",
              "VIT_MODEL_ID", "VIT_REVISION", "SPLIT_MODE", "SMOKE_TEST", "MODELS_TO_RUN",
              "EPOCHS_BASELINE", "EPOCHS_STAGE_A", "EPOCHS_STAGE_B", "UNFREEZE_LAST_N",
              "FREEZE_BATCHNORM", "CNN_AUGMENT", "FUSION_AUGMENT", "MIXUP_ALPHA",
              "PREPROCESS_VERSION", "SYNC_AUGMENT_COPIES", "LIGHT_MAX_MACRO_F1_DROP", "LIGHT_MIN_LABEL_AGREEMENT"]
CONFIG = {k: globals()[k] for k in CONFIG_KEYS}
write_json(RUN_DIR / "config.json", CONFIG)
ENVIRONMENT = {"python": sys.version, "platform": platform.platform(),
               "tensorflow": tf.__version__, "torch": torch.__version__,
               "transformers": transformers.__version__,
               "tf_gpus": [x.name for x in tf.config.list_physical_devices("GPU")],
               "torch_gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
write_json(RUN_DIR / "environment.json", ENVIRONMENT)
(RUN_DIR / "pip-freeze.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
print(ENVIRONMENT)

## 3 Kiểm tra ảnh và loại trùng trước chia tập
Hash byte phát hiện tệp giống hệt; hash pixel RGB phát hiện cùng nội dung giải mã dù metadata khác. Đây **không phải kiểm tra near-duplicate hoặc tách bệnh nhân**. Các ảnh khác lát cắt của cùng bệnh nhân vẫn có thể gây rò rỉ. Không xóa tệp nguồn: chỉ loại khỏi manifest của run. Nếu cùng ảnh mang hai nhãn khác nhau, dừng để kiểm tra.

Ảnh hỏng được ghi `invalid_images.json` rồi dừng. Hãy sửa dữ liệu và tạo run mới; không âm thầm bỏ ảnh rồi báo đủ số mẫu.

### Cách đọc kiểm tra dữ liệu
**Đầu vào:** ảnh trong ba thư mục lớp. **Đầu ra:** danh sách ảnh hợp lệ duy nhất, số mẫu từng lớp, `dataset_inventory.json`, `duplicates.json`, `invalid_images.json`.

Hash byte giống nhau nghĩa là tệp giống hệt; hash pixel giống nhau nghĩa là ảnh RGB giải mã giống nhau về kích thước và giá trị pixel. Near-duplicate là ảnh gần giống nhưng không hoàn toàn giống; bước này chưa giải quyết loại đó. Nếu ảnh trùng khác nhãn, cần kiểm tra nhãn gốc, không tự chọn nhãn bằng số đông.

Số mẫu phải đọc từ kết quả thực của run. Ảnh không đọc được làm dừng quy trình để tránh âm thầm thay đổi tập dữ liệu.

In [ ]:
def rgb_image(path):
    """Giữ cách decode RGB của nguồn; không tự xoay/crop tăng cường ảnh."""
    with Image.open(path) as im:
        return im.convert("RGB").copy()

def scan_dataset(base_dir):
    """Kiểm tra từng ảnh, dedup trong manifest và chặn nhãn xung đột."""
    rows, duplicates, invalid, seen = [], [], [], {}
    suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    for label, name in enumerate(CLASS_NAMES):
        folder = base_dir / name
        if not folder.is_dir(): raise FileNotFoundError(f"Thiếu thư mục lớp: {folder}")
        for path in sorted(p for p in folder.rglob("*") if p.suffix.lower() in suffixes):
            try:
                im = rgb_image(path)
                pixel_hash = hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()
                row = {"relative_path": str(path.relative_to(base_dir)), "label": label,
                       "class_name": name, "sha256": sha256_file(path),
                       "pixel_hash": pixel_hash, "width": im.width, "height": im.height}
                if pixel_hash in seen:
                    prior = seen[pixel_hash]
                    if prior["label"] != label:
                        raise ValueError(f"Ảnh trùng khác nhãn: {prior['relative_path']} và {path}")
                    duplicates.append({**row, "kept": prior["relative_path"]})
                else:
                    seen[pixel_hash] = row; rows.append(row)
            except ValueError:
                raise
            except Exception as error:
                invalid.append({"path": str(path), "error": repr(error)})
    write_json(RUN_DIR / "duplicates.json", duplicates)
    write_json(RUN_DIR / "invalid_images.json", invalid)
    if invalid: raise RuntimeError("Có ảnh không đọc được; xem invalid_images.json.")
    if not rows: raise RuntimeError("Không tìm thấy ảnh.")
    write_json(RUN_DIR / "dataset_inventory.json", rows)
    return rows

records = scan_dataset(BASE_DIR)
counts = {name: sum(r["class_name"] == name for r in records) for name in CLASS_NAMES}
print("Ảnh duy nhất sau làm sạch:", counts, "Tổng:", len(records))

## 4 Chia tập và khóa manifest
Tạo split trước augmentation. `SMOKE_TEST=True` lấy tối đa hai batch mỗi lớp mỗi tập sau khi split, chỉ để kiểm tra chức năng; mặc định False chạy toàn bộ dữ liệu hợp lệ. Số mẫu thực tế được ghi trong manifest; không dùng số liệu cũ để điền báo cáo.

### Vai trò của ba tập
| Tập | Vai trò | Có cập nhật trọng số không |
|---|---|---|
| Train | Dùng để học trọng số | Có |
| Validation | Theo dõi và chọn checkpoint/cấu hình | Không, nhưng ảnh hưởng quyết định chọn mô hình |
| Test | Đánh giá sau khi chốt mô hình | Không; không dùng chọn tham số |

Stratified split giữ tỷ lệ lớp gần tương đương giữa các tập, không giải quyết việc cùng bệnh nhân xuất hiện ở nhiều tập. Kiểm tra giao hash phải không có phần tử chung. **Đầu ra:** `split_manifest.json` và hash của manifest; mọi bảng kết quả sau phải dẫn được về split này. Sau dedup và làm tròn, số mẫu thực tế có thể không đúng chính xác 70/15/15 phần trăm.

In [ ]:
labels = [r["label"] for r in records]
train_rows, rest = train_test_split(records, train_size=0.70, stratify=labels, random_state=SEED)
# 30% còn lại chia đôi để tạo validation 15% và test 15%.
val_rows, test_rows = train_test_split(rest, test_size=0.5,
    stratify=[r["label"] for r in rest], random_state=SEED)
if SMOKE_TEST:
    def subset(rows):
        return [r for c in range(NUM_CLASSES) for r in
                [x for x in rows if x["label"] == c][:2 * BATCH_SIZE]]
    train_rows, val_rows, test_rows = map(subset, (train_rows, val_rows, test_rows))
splits = {"train": train_rows, "validation": val_rows, "test": test_rows}
for name, rows in splits.items():
    rows.sort(key=lambda r: r["relative_path"])
    if rows: assert set(r["label"] for r in rows) == set(range(NUM_CLASSES)), f"Thiếu lớp: {name}"
for key in ("sha256", "pixel_hash"):
    sets = [{r[key] for r in rows} for rows in splits.values()]
    assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])
write_json(RUN_DIR / "split_manifest.json", splits)
SPLIT_HASH = sha256_file(RUN_DIR / "split_manifest.json")
print({name: len(rows) for name, rows in splits.items()}, "split hash:", SPLIT_HASH)
total_used = sum(len(rows) for rows in splits.values())
print("Tỷ lệ thực tế sau làm tròn:", {name: round(100*len(rows)/total_used, 2) for name, rows in splits.items()})
write_json(RUN_DIR / "split_summary.json", {"requested_ratios": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "actual_counts": {name: len(rows) for name, rows in splits.items()},
    "actual_ratios": {name: len(rows)/total_used for name, rows in splits.items()},
    "class_counts": {name: {c: sum(r["class_name"]==c for r in rows) for c in CLASS_NAMES} for name, rows in splits.items()},
    "smoke": SMOKE_TEST})
# TODO [DATA-01]: Bổ sung patient ID từ nguồn đáng tin và chia theo bệnh nhân trước kết luận tổng quát hóa.

## 4A Tăng cường dữ liệu train và giảm nguy cơ học thuộc

Chúng tôi giữ nguyên split gốc 70/15/15 và chỉ sinh biến thể **sau khi chia tập**, từ ảnh thuộc train. Mặc định mỗi ảnh train có thêm một biến thể xoay nhẹ trong ±8 độ, tịnh tiến tối đa 3% mỗi chiều và thay đổi độ sáng trong khoảng 0.95–1.05. Không flip, crop hoặc biến đổi validation/test. Các mức này là cấu hình kỹ thuật cần đánh giá trên validation, không phải đảm bảo phù hợp mọi ảnh y khoa.

Quan trọng: ảnh đã biến đổi được lưu thành một mẫu train mới; **CNN và ViT đều đọc đúng cùng tệp đó**. Không dùng CLS của ảnh gốc ghép với ảnh đã augmentation. Lưu parent hash để biết biến thể sinh từ đâu. Tổng mẫu train sau augmentation lớn hơn 70% số ảnh gốc; tỷ lệ 70/15/15 vẫn nói về các ảnh gốc duy nhất trước augmentation.

Augmentation, Dropout, L2/AdamW, đóng băng BatchNorm và chọn best bằng validation giúp kiểm soát overfitting nhưng không bảo đảm ảnh ngoài dataset được dự đoán tốt. Muốn chứng minh khả năng đó cần tập ảnh ngoài độc lập có nhãn chuẩn; không dùng những ảnh chỉ chọn bằng mắt để tính độ chính xác. Cùng bệnh nhân/nguồn chụp giữa các tập vẫn là hạn chế nếu chưa có metadata để loại trừ.


In [ ]:
from PIL import ImageEnhance
# Luôn lấy ảnh gốc từ split đã khóa; chạy lại cell không nhân đôi augmentation.
original_train_rows = list(splits["train"])
heldout_hashes = {r["sha256"] for r in val_rows + test_rows}
aug_dir = RUN_DIR / "train_augmentation"
aug_dir.mkdir(exist_ok=True)
expanded = list(original_train_rows)
known_pixels = {r["pixel_hash"] for r in original_train_rows + val_rows + test_rows}
skipped_augmentations = []
for row in original_train_rows:
    assert row["sha256"] not in heldout_hashes
    seed = int(hashlib.sha256((row["sha256"] + str(SEED)).encode()).hexdigest()[:16], 16)
    rng = np.random.default_rng(seed)
    for copy_id in range(SYNC_AUGMENT_COPIES):
        image = rgb_image(BASE_DIR / row["relative_path"])
        angle = float(rng.uniform(-8, 8))
        shift = (float(rng.uniform(-0.03, 0.03) * image.width),
                 float(rng.uniform(-0.03, 0.03) * image.height))
        brightness = float(rng.uniform(0.95, 1.05))
        border = tuple(int(v) for v in np.asarray(image, dtype=np.uint8).reshape(-1,3).mean(axis=0))
        variant = image.rotate(angle, resample=Image.Resampling.BILINEAR,
                               translate=shift, fillcolor=border)
        variant = ImageEnhance.Brightness(variant).enhance(brightness)
        path = aug_dir / f"{row['sha256']}_{copy_id}.png"
        variant.save(path)
        byte_hash = sha256_file(path)
        variant_pixel_hash = hashlib.sha256(str(variant.size).encode()+variant.tobytes()).hexdigest()
        if byte_hash in heldout_hashes or variant_pixel_hash in known_pixels:
            skipped_augmentations.append({"parent_sha256": row["sha256"], "copy_id": copy_id, "reason": "exact_duplicate"})
            path.unlink()
            continue
        known_pixels.add(variant_pixel_hash)
        expanded.append({**row, "relative_path": str(path.resolve()), "sha256": byte_hash,
            "pixel_hash": variant_pixel_hash,
            "parent_sha256": row["sha256"], "augmentation": {"rotation_degrees": angle,
            "translate_pixels": shift, "brightness": brightness}})
train_rows = expanded
FEATURE_ROWS = {"train": train_rows, "validation": val_rows, "test": test_rows}
# Manifest split gốc không bị sửa; manifest này ghi các mẫu thực dùng train.
write_json(RUN_DIR / "training_samples_manifest.json", train_rows)
write_json(RUN_DIR / "augmentation_summary.json", {"original_train_images": len(original_train_rows),
    "training_samples_including_augmented": len(train_rows), "requested_copies_per_image": SYNC_AUGMENT_COPIES, "skipped_exact_duplicates": skipped_augmentations,
    "validation_images_unchanged": len(val_rows), "test_images_unchanged": len(test_rows),
    "policy": "train-only offline synchronized CNN/ViT; rotation8deg/translation3pct/brightness5pct"})
print("Ảnh train gốc:", len(original_train_rows), "Mẫu train sau augmentation:", len(train_rows))


## 5 Trích đặc trưng ViT và kiểm chứng cache
ViT pretrained được đóng băng, chỉ lấy token CLS 768 chiều. `ViT_Head` chỉ train head, không fine-tune toàn ViT. Cache được ghép đúng thứ tự ảnh; dùng `allow_pickle=False`. Resolve revision thành commit và ghi lại; muốn tái chạy cùng pretrained thì dùng commit đó thay `main`.

Giải phóng ViT khỏi GPU sau trích train/validation; test chỉ được trích ở bước test cuối. Điều này giảm tranh chấp GPU, không làm mô hình điện thoại nhẹ hơn.

### Dữ liệu đi qua ViT như thế nào
Ảnh RGB → resize theo cấu hình → processor chuẩn hóa → ViT pretrained → vector CLS 768 chiều. ViT ở chế độ evaluation và không tính gradient trong bước này; chỉ head/fusion được train về sau.

**Đầu vào:** manifest train/validation và pretrained commit đã resolve. **Đầu ra:** hai ma trận đặc trưng có dạng `(số ảnh, 768)`, cache có metadata, `vit_source.json` và `vit_processor.json`. Số hàng phải bằng số ảnh; dòng thứ k phải thuộc đúng ảnh thứ k trong manifest.

Một feature vector không phải ảnh hoặc nhãn. Việc lưu vector giúp train head nhanh hơn nhưng không loại bỏ nhu cầu chạy ViT khi đưa ảnh mới vào hệ thống. Khi triển khai điện thoại vẫn phải có ViT hoặc một phương án thay thế được nghiên cứu và đánh giá riêng.

In [ ]:
RESOLVED_VIT_REVISION = model_info(VIT_MODEL_ID, revision=VIT_REVISION).sha
if not RESOLVED_VIT_REVISION: raise RuntimeError("Không resolve được commit ViT.")
processor = ViTImageProcessor.from_pretrained(VIT_MODEL_ID, revision=RESOLVED_VIT_REVISION)
write_json(RUN_DIR / "vit_processor.json", processor.to_dict())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_vit = None

def get_vit():
    global _vit
    if _vit is None:
        _vit = ViTModel.from_pretrained(VIT_MODEL_ID, revision=RESOLVED_VIT_REVISION).to(DEVICE).eval()
        assert _vit.config.hidden_size == 768, "Builder hiện dùng feature 768 chiều."
    return _vit

def release_vit():
    global _vit
    _vit = None
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def vit_batch(paths):
    """Trích CLS từ ảnh mới bằng đúng pipeline đã dùng tạo cache."""
    images = [rgb_image(p).resize(VIT_SIZE, Image.Resampling.NEAREST) for p in paths]
    inputs = processor(images=images, return_tensors="pt")
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        output = get_vit()(**inputs).last_hidden_state[:, 0, :]
    return output.cpu().numpy().astype(np.float32)

def extract_features(rows):
    """Chỉ tái dùng cache khi ảnh, thứ tự, processor và môi trường đều khớp."""
    identity = {"images": [(r["relative_path"], r["sha256"]) for r in rows],
                "vit": VIT_MODEL_ID, "revision": RESOLVED_VIT_REVISION,
                "processor": processor.to_dict(), "resize": [*VIT_SIZE, "nearest"],
                "preprocessing": PREPROCESS_VERSION, "environment": ENVIRONMENT}
    key = digest_json(identity)
    array_path, meta_path = CACHE_ROOT / f"{key}.npy", CACHE_ROOT / f"{key}.json"
    if array_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text())
        feats = np.load(array_path, allow_pickle=False)
        if (meta.get("identity_hash") == key and meta.get("array_sha256") == sha256_file(array_path)
                and feats.shape == (len(rows), 768) and np.isfinite(feats).all()):
            print("Dùng cache đã đối chiếu:", key[:12]); return feats
    feats = np.concatenate([vit_batch([BASE_DIR / r["relative_path"] for r in rows[i:i+VIT_BATCH_SIZE]])
                            for i in range(0, len(rows), VIT_BATCH_SIZE)])
    assert feats.shape == (len(rows), 768) and np.isfinite(feats).all()
    np.save(array_path, feats, allow_pickle=False)
    write_json(meta_path, {"identity_hash": key, "identity": identity,
                          "array_sha256": sha256_file(array_path)})
    return feats

# Train dùng manifest sau augmentation; split gốc chỉ mô tả cách chia 70/15/15.
FEATURE_ROWS = {"train": train_rows, "validation": val_rows, "test": test_rows}
features = {name: extract_features(FEATURE_ROWS[name]) for name in ("train", "validation")}
FEATURE_ROW_IDS = {name: [(r["relative_path"], r["sha256"]) for r in FEATURE_ROWS[name]]
                   for name in ("train", "validation")}
for name in ("train", "validation"):
    assert len(FEATURE_ROWS[name]) == features[name].shape[0], f"Lệch số mẫu/CLS: {name}"
release_vit()
# TF giữ VRAM đã cấp phát sau train; dùng CPU cho ViT ở các bước ảnh đơn/test/export sau đó.
DEVICE = "cpu"
FEATURE_DIM = 768
write_json(RUN_DIR / "vit_source.json", {"model_id": VIT_MODEL_ID, "revision": RESOLVED_VIT_REVISION})

## 6 Bộ nạp dữ liệu
Mỗi batch trả ảnh, CLS hoặc cặp ảnh/CLS tùy kiến trúc. Validation/test không shuffle và không augmentation. Mặc định augmentation online của CNN và fusion đều tắt. Augmentation offline được tạo ở mục 4A; CLS được tính từ chính từng ảnh đã biến đổi, dùng cùng một manifest cho cả bốn mô hình. Muốn augmentation đồng bộ phải trích lại ViT cho ảnh đã biến đổi và ghi thành protocol khác.

Class weights chỉ tính từ train. Không trộn MixUp với trọng số theo nhãn cứng trong bản này.

### Cách ghép một batch
Với CNN, đầu vào là tensor ảnh. Với ViT Head, đầu vào là CLS. Với fusion, đầu vào là cặp `(ảnh, CLS)` của cùng mẫu. Nhãn dùng one-hot: Benign `[1,0,0]`, Malignant `[0,1,0]`, Normal `[0,0,1]`.

Augmentation tạo biến thể ảnh train; không áp vào validation/test. Class weight tăng mức ảnh hưởng của lớp ít mẫu trong loss, không tạo thêm ảnh và không bảo đảm recall tăng. **Đầu ra:** bộ nạp đúng kiểu đầu vào của từng model và `class_weights.json`. Cần kiểm tra parity ảnh/CLS nếu thay pipeline.

In [ ]:
augmenter = ImageDataGenerator(rotation_range=25, width_shift_range=0.20,
    height_shift_range=0.20, shear_range=0.15, zoom_range=0.20,
    horizontal_flip=True, brightness_range=[0.85, 1.15], channel_shift_range=15)

def cnn_image(path, augment=False):
    """RGB [0,1], resize nearest giống load_img mặc định trong nguồn."""
    image = np.array(rgb_image(path).resize(IMG_SIZE, Image.Resampling.NEAREST), dtype=np.float32)
    if augment: image = augmenter.random_transform(image)
    return image.astype(np.float32) / 255.0

class DataSequence(tf.keras.utils.Sequence):
    """Ghép nhãn, ảnh và CLS theo cùng chỉ số; không shuffle khi đánh giá."""
    def __init__(self, rows, feats, mode, training=False, **kwargs):
        super().__init__(**kwargs)
        self.rows, self.feats, self.mode, self.training = rows, feats, mode, training
        self.indices = np.arange(len(rows)); self.rng = np.random.default_rng(SEED)
        if len(rows) != len(feats):
            raise ValueError(f"Số ảnh ({len(rows)}) khác số CLS ({len(feats)}). "
                             "Trích lại feature từ train_rows sau augmentation; không cắt mảng để khớp.")
        if np.asarray(feats).shape != (len(rows), FEATURE_DIM):
            raise ValueError("Feature phải có dạng (số ảnh, 768).")
        self.on_epoch_end()
    def __len__(self): return math.ceil(len(self.rows) / BATCH_SIZE)
    def on_epoch_end(self):
        if self.training: self.rng.shuffle(self.indices)
    def __getitem__(self, index):
        ids = self.indices[index * BATCH_SIZE:(index+1) * BATCH_SIZE]
        y = tf.keras.utils.to_categorical([self.rows[k]["label"] for k in ids], NUM_CLASSES)
        if self.mode == "vit": return self.feats[ids], y
        augment = self.training and (CNN_AUGMENT if self.mode == "cnn" else FUSION_AUGMENT)
        images = np.stack([cnn_image(BASE_DIR / self.rows[k]["relative_path"], augment) for k in ids])
        return (images if self.mode == "cnn" else (images, self.feats[ids])), y

cw = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES),
                          y=[r["label"] for r in train_rows])
class_weights = {i: float(weight) for i, weight in enumerate(cw)}
write_json(RUN_DIR / "class_weights.json", class_weights)
print(class_weights)

## 7 Kiến trúc và serialization
Giữ các lớp/chiều của bốn builder nguồn: CNN, ViT head, CNN–ViT nối đặc trưng và Adaptive AACNN–ViT. Attention trong bản nguồn là **self-attention trên hai token tổng hợp CNN/ViT**, không phải attention trên toàn patch ảnh. AdaptiveGate học một hệ số trộn mỗi mẫu.

Thay Lambda bằng layer có tên/cấu hình. ResNet được nối phẳng vào graph để lấy activation và gradient Grad-CAM; không đổi phép tính dự kiến. Cần kiểm tra parity với checkpoint nguồn nếu có; chưa khẳng định weight naming tương thích checkpoint cũ. Notebook mới train checkpoint mới.

### Bốn mô hình đang được huấn luyện
| Mô hình | Đầu vào | Cách tạo kết quả |
|---|---|---|
| CNN | Ảnh RGB | Ba convolution và head |
| ViT Head | CLS ViT pretrained | Head học phân loại từ vector cố định |
| CNN–ViT | Ảnh và CLS | CNN nhỏ, nối đặc trưng, head |
| Adaptive AACNN–ViT | Ảnh và CLS | ResNet50, token projection, adaptive gate, attention hai token và head |

ResNet50 là backbone CNN pretrained. Global average pooling tóm tắt mỗi kênh đặc trưng thành một số. Projection đưa hai nhánh về cùng chiều token. Adaptive gate học alpha để trộn hai token. Multi-head attention cho hai token trao đổi thông tin; LayerNorm ổn định biểu diễn, Dropout bỏ ngẫu nhiên một phần activation trong train, L2 thêm phạt trọng số lớn vào loss.

Serialization là ghi cấu trúc/cấu hình để nạp lại. Việc thay Lambda bằng layer có tên nhằm hỗ trợ lưu/nạp, chưa chứng minh checkpoint cũ nạp được vào graph mới. **Đầu ra ở bước này là các hàm dựng model, chưa có trọng số đã train.**

In [ ]:
def build_cnn_baseline():
    """CNN baseline ba convolution; đầu vào RGB [0,1], không dùng ResNet pretrained."""
    inp = layers.Input(shape=(*IMG_SIZE,3))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inp, out, name="CNN_Baseline")


# --- ViT head (feature-only) ---
def build_vit_head():
    """Head phân loại trên CLS ViT đóng băng; không fine-tune ViT trong graph Keras."""
    inp = layers.Input(shape=(FEATURE_DIM,))
    x = layers.Dense(256, activation="relu")(inp)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.35)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inp, out, name="ViT_Head")


# --- CNN-ViT baseline (concat) ---
def build_cnn_vit():
    """Nối đặc trưng CNN nhỏ và CLS ViT; giữ chiều/lớp của bản nguồn."""
    img_in = layers.Input(shape=(*IMG_SIZE,3), name="image_input")
    vit_in = layers.Input(shape=(FEATURE_DIM,), name="vit_features_input")

    x = layers.Conv2D(16, 3, padding="same", activation="relu")(img_in)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    cnn_feat = layers.GlobalAveragePooling2D(name="cnn_gap")(x)

    fused = layers.Concatenate()([cnn_feat, vit_in])
    h = layers.Dense(256, activation="relu")(fused)
    h = layers.Dropout(0.5)(h)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(h)

    return tf.keras.Model([img_in, vit_in], out, name="CNN_ViT")

In [ ]:
@tf.keras.utils.register_keras_serializable(package="LungCT")
class ResNetPreprocess(layers.Layer):
    """Chuyển RGB [0,1] sang preprocessing ResNet ImageNet đúng một lần."""
    def call(self, x): return tf.keras.applications.resnet.preprocess_input(x * 255.0)

@tf.keras.utils.register_keras_serializable(package="LungCT")
class StackTokens(layers.Layer):
    def call(self, inputs): return tf.stack(inputs, axis=1)
    def compute_output_shape(self, shapes): return (shapes[0][0], len(shapes), shapes[0][-1])

@tf.keras.utils.register_keras_serializable(package="LungCT")
class AdaptiveGate(layers.Layer):
    """Học alpha trong [0,1] để trộn hai token có cùng chiều."""
    def __init__(self, d_hidden=128, dropout=0.25, l2=1e-4, **kwargs):
        super().__init__(**kwargs)
        self.d_hidden, self.dropout_rate, self.l2_rate = d_hidden, dropout, l2
        self.concat = layers.Concatenate()
        self.d1 = layers.Dense(d_hidden, activation="relu", kernel_regularizer=regularizers.l2(l2))
        self.drop = layers.Dropout(dropout)
        self.logit = layers.Dense(1, kernel_initializer="zeros", bias_initializer="zeros")
    def build(self, input_shape):
        # Tạo biến rõ ràng để Keras khôi phục sublayer weights an toàn.
        a, b = input_shape
        if a[-1] != b[-1]: raise ValueError("Hai token phải cùng chiều.")
        self.concat.build(input_shape)
        self.d1.build((a[0], a[-1] + b[-1]))
        self.drop.build((a[0], self.d_hidden))
        self.logit.build((a[0], self.d_hidden))
        super().build(input_shape)
    def call(self, inputs, training=None):
        a, b = inputs
        alpha = tf.sigmoid(self.logit(self.drop(self.d1(self.concat([a, b])), training=training)))
        return alpha * a + (1-alpha) * b
    def get_config(self):
        return {**super().get_config(), "d_hidden": self.d_hidden,
                "dropout": self.dropout_rate, "l2": self.l2_rate}

def _resnet_backbone(img_in):
    # Graph phẳng giúp truy cập convolution của ResNet từ model cuối để lấy gradient.
    x_img = ResNetPreprocess(name="resnet_preprocess")(img_in)
    base = tf.keras.applications.ResNet50(include_top=False, weights="imagenet",
        input_tensor=x_img, pooling="avg")
    base.trainable = False
    return base, base.output

In [ ]:
def build_aacnn_adaptive(d_model=256, l2=1e-4):
    """ResNet50 và CLS ViT qua adaptive gate cùng attention hai token; giữ kiến trúc nguồn."""
    img_in = layers.Input(shape=(*IMG_SIZE,3), name="image_input")
    vit_in = layers.Input(shape=(FEATURE_DIM,), name="vit_features_input")
    base, cnn_feat = _resnet_backbone(img_in)

    cnn_feat = layers.Dense(512, activation="relu")(cnn_feat)
    cnn_feat = layers.Dropout(0.25)(cnn_feat)
    cnn_tok = layers.Dense(d_model, activation="relu", kernel_regularizer=regularizers.l2(l2), name="cnn_token")(cnn_feat)

    vit_feat = layers.LayerNormalization()(vit_in)
    vit_feat = layers.Dense(512, activation="gelu")(vit_feat)
    vit_feat = layers.Dropout(0.25)(vit_feat)
    vit_tok = layers.Dense(d_model, activation="gelu", kernel_regularizer=regularizers.l2(l2), name="vit_token")(vit_feat)

    gated = AdaptiveGate(name="adaptive_gate")([cnn_tok, vit_tok])

    tokens = StackTokens(name="stack_tokens")([cnn_tok, vit_tok])  # [B,2,D]
    attn = layers.MultiHeadAttention(num_heads=4, key_dim=d_model//4, dropout=0.10, name="cross_attn")
    attn_out = attn(tokens, tokens)
    attn_out = layers.Add()([tokens, attn_out])
    attn_out = layers.LayerNormalization()(attn_out)
    attn_flat = layers.Flatten(name="attn_flat")(attn_out)

    fused = layers.Concatenate(name="fusion_concat")([gated, cnn_tok, vit_tok, attn_flat])

    h = layers.Dense(1024, activation="gelu", kernel_regularizer=regularizers.l2(l2))(fused)
    h = layers.Dropout(0.40)(h)
    h = layers.Dense(512, activation="gelu", kernel_regularizer=regularizers.l2(l2))(h)
    h = layers.Dropout(0.35)(h)
    h = layers.Dense(256, activation="gelu")(h)
    h = layers.Dropout(0.30)(h)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="classifier")(h)

    model = tf.keras.Model([img_in, vit_in], out, name="AACNN_ViT_ADAPTIVE")
    model._cnn_backbone = base
    return model

## 8 Loss và checkpoint toàn quá trình
Giữ hạng focal/CCE và tỷ lệ 0.7/0.3 của nguồn, nhưng trả loss theo từng mẫu để áp class_weight đúng từng ảnh. Đây là sửa train semantics, không phải tái lập loss có trọng số nguyên trạng. Công thức này không tự động tương đương mọi định nghĩa focal loss; không tuyên bố tối ưu mất cân bằng nếu chưa so sánh.

Macro F1 được tính trên toàn validation sau mỗi epoch và dùng chọn checkpoint chung cho cả stage A/B. Stage B bắt đầu từ best A, giảm learning rate và mặc định giữ BatchNorm frozen. Đây là thay đổi train policy. Không tự động thực hiện No-Adaptive vì nguồn chưa định nghĩa mô hình đó.

### Cách đọc quá trình học
Loss là đại lượng optimizer cố giảm; accuracy là tỷ lệ dự đoán đúng. Macro F1 lấy trung bình F1 các lớp, để lớp ít mẫu không bị lớp đông hoàn toàn lấn át. F1 kết hợp precision và recall; macro F1 cao hơn không tự chứng minh an toàn lâm sàng.

Stage A đóng băng ResNet để train phần phía sau. Stage B mở một phần layer cuối với learning rate thấp hơn; mặc định giữ BatchNorm đóng băng. AdamW cập nhật trọng số; early stopping dừng khi validation không cải thiện; giảm learning rate khi validation loss chững lại.

**Đầu ra khi train:** `best.weights.h5`, `history.json`, `training.json`, `summary.txt`. `best.weights.h5` giữ trọng số có validation macro F1 tốt nhất xuyên hai stage; không mặc định epoch cuối hoặc stage B tốt hơn stage A.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="LungCT")
def source_hybrid_loss(y_true, y_pred):
    """Giữ hạng focal/CCE nguồn; trả vector per-sample để áp class weights."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    pt = y_true * y_pred + (1-y_true) * (1-y_pred)
    focal = tf.reduce_mean(0.25 * tf.pow(1-pt, 2.0) * (-y_true) * tf.math.log(y_pred), axis=-1)
    # Cùng trung bình không trọng số như nguồn; vector theo mẫu sửa áp class_weight đúng mỗi mẫu.
    return 0.7 * focal + 0.3 * ce

# Thay scalar batch bằng vector per-sample là sửa lỗi trọng số; ghi trong manifest, không gọi loss parity tuyệt đối.
def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=5e-6),
                  loss=source_hybrid_loss, metrics=["accuracy"])

def predict_sequence(model, seq):
    """Trả nhãn/scores đúng thứ tự; không gọi on_epoch_end ở giữa đánh giá."""
    ys, probs = [], []
    for i in range(len(seq)):
        x, y = seq[i]
        probs.append(np.asarray(model(x, training=False))); ys.append(np.asarray(y))
    return np.concatenate(ys), np.concatenate(probs)

class ValidationCheckpoint(tf.keras.callbacks.Callback):
    """Giữ best macro F1 xuyên hai stage; lưu history theo epoch xuống Drive."""
    def __init__(self, seq, path):
        super().__init__(); self.seq, self.path = seq, Path(path)
        self.best, self.rows = -1.0, []
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        yt, yp = predict_sequence(self.model, self.seq)
        score = float(f1_score(yt.argmax(1), yp.argmax(1), labels=np.arange(NUM_CLASSES),
                               average="macro", zero_division=0))
        logs["val_macro_f1"] = score
        self.rows.append({"global_epoch": len(self.rows)+1,
                          **{k: float(v) for k, v in logs.items()}})
        if score > self.best:
            self.best = score; self.model.save_weights(self.path)
        write_json(self.path.parent / "history.json", self.rows)
        print(f" — val_macro_f1={score:.4f}; best={self.best:.4f}")

BUILDERS = {"CNN": build_cnn_baseline, "ViT_Head": build_vit_head,
            "CNN_ViT": build_cnn_vit, "AACNN_Adaptive": build_aacnn_adaptive}
MODES = {"CNN": "cnn", "ViT_Head": "vit", "CNN_ViT": "fusion", "AACNN_Adaptive": "fusion"}

def train_one(name):
    """Train một model mới; chỉ chọn bằng validation, trả đường dẫn best weights."""
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(SEED)
    model = BUILDERS[name]()
    model_dir = RUN_DIR / name; model_dir.mkdir(exist_ok=True)
    (model_dir / "summary.txt").write_text("", encoding="utf-8")
    with open(model_dir / "summary.txt", "w") as stream: model.summary(print_fn=lambda s, **k: stream.write(s+"\n"))
    train_seq = DataSequence(train_rows, features["train"], MODES[name], training=True)
    val_seq = DataSequence(val_rows, features["validation"], MODES[name])
    tracker = ValidationCheckpoint(val_seq, model_dir / "best.weights.h5")
    def callbacks():
        return [tracker,
                tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=7),
                tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
                tf.keras.callbacks.TerminateOnNaN()]
    compile_model(model, 2e-4 if name == "AACNN_Adaptive" else 1e-4)
    epochs = 1 if SMOKE_TEST else (EPOCHS_STAGE_A if name == "AACNN_Adaptive" else EPOCHS_BASELINE)
    started = time.perf_counter()
    model.fit(train_seq, validation_data=val_seq, epochs=epochs,
              class_weight=class_weights, callbacks=callbacks(), verbose=1)
    if not tracker.path.exists(): raise RuntimeError("Không có checkpoint hợp lệ sau stage A.")
    model.load_weights(tracker.path)
    if name == "AACNN_Adaptive":
        backbone = model._cnn_backbone
        backbone.trainable = True
        for layer in backbone.layers: layer.trainable = False
        for layer in backbone.layers[-UNFREEZE_LAST_N:]:
            layer.trainable = not (FREEZE_BATCHNORM and isinstance(layer, layers.BatchNormalization))
        compile_model(model, 5e-5)
        model.fit(train_seq, validation_data=val_seq, epochs=1 if SMOKE_TEST else EPOCHS_STAGE_B,
                  class_weight=class_weights, callbacks=callbacks(), verbose=1)
        model.load_weights(tracker.path)
    write_json(model_dir / "training.json", {"wall_seconds": time.perf_counter()-started,
        "best_val_macro_f1": tracker.best, "parameters": model.count_params(),
        "checkpoint_sha256": sha256_file(tracker.path), "smoke": SMOKE_TEST})
    return model, model_dir, val_seq

## 8A Kiểm tra kỹ thuật trước lượt train dài

Bước kiểm tra này dùng các model tạm dựng mới, một batch train và một batch validation để phát hiện lỗi hợp đồng đầu vào, `fit`, class weight và serialization trước khi đầu tư toàn bộ epoch. Không dùng test. Các trọng số tạm bị bỏ; model thực nghiệm được dựng lại từ đầu sau đó. Đây không phải chế độ rút gọn kết quả nghiên cứu.

Nếu một kiểm tra thất bại, quy trình dừng và lưu lỗi. `preflight.json` chỉ chứng minh các thao tác kỹ thuật đã chạy trong môi trường hiện tại, không xác nhận độ chính xác hoặc khả năng chạy trên điện thoại. Bản train chính thức vẫn dùng toàn bộ dữ liệu và ngân sách cấu hình.


In [ ]:
# Chặn cả trường hợp số lượng bằng nhau nhưng thứ tự ảnh khác.
for split_name, current_rows in (("train", train_rows), ("validation", val_rows)):
    current_ids = [(r["relative_path"], r["sha256"]) for r in current_rows]
    if FEATURE_ROW_IDS[split_name] != current_ids:
        raise ValueError(f"Manifest {split_name} đổi sau trích CLS; chạy lại mục 5 trước preflight.")

preflight_results = {}
for name in MODELS_TO_RUN:
    tf.keras.backend.clear_session(); gc.collect()
    temporary_model = BUILDERS[name]()
    temporary_seq = DataSequence(train_rows, features["train"], MODES[name], training=True)
    temporary_val = DataSequence(val_rows, features["validation"], MODES[name])
    preflight_dir = RUN_DIR / "preflight" / name
    preflight_dir.mkdir(parents=True, exist_ok=True)
    try:
        compile_model(temporary_model, 1e-4)
        temporary_model.fit(temporary_seq, epochs=1, steps_per_epoch=1,
                            class_weight=class_weights, verbose=0)
        probe, _ = temporary_val[0]
        expected = np.asarray(temporary_model(probe, training=False))
        temporary_model.save(preflight_dir / "probe.keras")
        loaded_probe = tf.keras.models.load_model(preflight_dir / "probe.keras", compile=False, safe_mode=True)
        actual = np.asarray(loaded_probe(probe, training=False))
        np.testing.assert_allclose(expected, actual, atol=1e-5, rtol=1e-4)
        preflight_results[name] = {"status": "PASS", "max_abs_error": float(np.max(np.abs(expected-actual)))}
        del loaded_probe
        # Model tạm không phải checkpoint nghiên cứu, xóa để giảm dung lượng Drive.
        (preflight_dir / "probe.keras").unlink()
    except Exception as error:
        preflight_results[name] = {"status": "FAIL", "error": repr(error)}
        write_json(RUN_DIR / "preflight.json", preflight_results)
        raise
    finally:
        del temporary_model, temporary_seq, temporary_val
        tf.keras.backend.clear_session(); gc.collect()
    write_json(RUN_DIR / "preflight.json", preflight_results)
print("Preflight:", preflight_results)


## 9 Đánh giá validation và biểu đồ không huấn luyện lại
Lưu báo cáo từng lớp, macro/weighted F1, ma trận nhầm lẫn thô và scores từng ảnh. ROC chỉ vẽ khi có cả mẫu dương và âm. Scores softmax là điểm mô hình, chưa chứng minh xác suất mắc bệnh đã được hiệu chuẩn.

Các biến mô hình được giải phóng lần lượt để giảm RAM. Đầu ra full `.keras` chỉ xuất sau kiểm tra nạp lại an toàn, không dùng `safe_mode=False`.

### Cách đọc kết quả
- Precision của lớp: trong các ảnh dự đoán là lớp đó, bao nhiêu ảnh đúng.
- Recall của lớp: trong các ảnh thật sự thuộc lớp đó, bao nhiêu ảnh được nhận đúng.
- F1 của lớp: trung bình điều hòa precision/recall; `support` là số ảnh thật của lớp.
- Confusion matrix: hàng là nhãn thật, cột là nhãn dự đoán. Ngoài đường chéo là nhầm lớp.
- ROC/AUC: đánh giá khả năng xếp hạng score từng lớp so với các lớp còn lại; khác accuracy tại argmax.

**Đầu ra:** metrics JSON, predictions CSV, ma trận thô, hình confusion matrix/ROC/history và model `.keras`. Roundtrip PASS chỉ có nghĩa output mẫu khớp sau lưu/nạp trong dung sai, không phải xác nhận toàn bộ chức năng hoặc chất lượng mô hình.

In [ ]:
def evaluate_and_save(model, seq, rows, directory, tag):
    """Lưu số đo và dữ liệu gốc; không gọi fit từ hàm đánh giá/vẽ."""
    directory = Path(directory)
    y, prob = predict_sequence(model, seq)
    actual, pred = y.argmax(1), prob.argmax(1)
    assert len(prob) == len(rows) and np.isfinite(prob).all()
    assert np.allclose(prob.sum(1), 1, atol=1e-4)
    report = classification_report(actual, pred, labels=np.arange(NUM_CLASSES),
        target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    cm = confusion_matrix(actual, pred, labels=np.arange(NUM_CLASSES))
    write_json(directory / f"{tag}_metrics.json", report)
    write_json(directory / f"{tag}_confusion_matrix.json", cm.tolist())
    with open(directory / f"{tag}_predictions.csv", "w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.writer(stream)
        writer.writerow(["image_sha256", "actual", "predicted", *[f"score_{n}" for n in CLASS_NAMES]])
        for r, a, b, scores in zip(rows, actual, pred, prob):
            writer.writerow([r["sha256"], CLASS_NAMES[a], CLASS_NAMES[b], *scores.tolist()])
    fig, ax = plt.subplots(figsize=(6, 5)); im = ax.imshow(cm, cmap="Blues")
    ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES), xticklabels=CLASS_NAMES,
           yticklabels=CLASS_NAMES, xlabel="Dự đoán", ylabel="Nhãn thật", title=f"{tag} — confusion matrix")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES): ax.text(j, i, str(cm[i,j]), ha="center", va="center")
    fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(directory/f"{tag}_cm.png", dpi=160); plt.show(); plt.close(fig)
    fig, ax = plt.subplots(figsize=(6,5))
    for i, name in enumerate(CLASS_NAMES):
        if len(np.unique(y[:, i])) < 2: continue
        fpr, tpr, _ = roc_curve(y[:, i], prob[:, i])
        ax.plot(fpr, tpr, label=f"{name} AUC={auc(fpr,tpr):.3f}")
    ax.plot([0,1],[0,1], "--", color="gray"); ax.set(xlabel="FPR", ylabel="TPR", title=f"{tag} ROC")
    ax.legend(); fig.tight_layout(); fig.savefig(directory/f"{tag}_roc.png",dpi=160); plt.close(fig)
    return report

def plot_history(model_dir):
    history = json.loads((model_dir / "history.json").read_text())
    fig, axes = plt.subplots(1, 3, figsize=(15,4))
    for ax, keys in zip(axes, [("accuracy","val_accuracy"), ("loss","val_loss"), ("val_macro_f1",)]):
        for key in keys: ax.plot([r.get(key, np.nan) for r in history], label=key)
        ax.set_xlabel("Epoch toàn run"); ax.legend(); ax.grid(alpha=0.2)
    fig.tight_layout(); fig.savefig(model_dir/"history.png",dpi=160); plt.show(); plt.close(fig)

validation_results = {}
for name in MODELS_TO_RUN:
    if name not in BUILDERS: raise ValueError(f"Chưa có builder: {name}")
    model, model_dir, val_seq = train_one(name)
    validation_results[name] = evaluate_and_save(model, val_seq, val_rows, model_dir, "validation")
    plot_history(model_dir)
    # Kiểm tra lưu/nạp full model trước khi dùng làm artifact triển khai.
    sample, _ = val_seq[0]
    before = np.asarray(model(sample, training=False))
    try:
        model.save(model_dir / "model.keras")
        reloaded = tf.keras.models.load_model(model_dir / "model.keras", compile=False, safe_mode=True)
        after = np.asarray(reloaded(sample, training=False))
        np.testing.assert_allclose(before, after, rtol=1e-4, atol=1e-5)
        write_json(model_dir / "roundtrip.json", {"status": "PASS", "max_abs_error": float(np.max(np.abs(before-after)))})
        del reloaded
    except Exception as error:
        write_json(model_dir / "roundtrip.json", {"status": "FAIL", "error": repr(error)})
        raise RuntimeError("Nạp lại model lỗi; xem roundtrip.json, chưa đủ điều kiện artifact.") from error
    del model, val_seq; tf.keras.backend.clear_session(); gc.collect()
write_json(RUN_DIR / "validation_comparison.json", validation_results)
print({name: result["macro avg"]["f1-score"] for name, result in validation_results.items()})

## 10 Nạp checkpoint và dự đoán một ảnh mới
Cố định mô hình đánh giá trước khi xem test; nếu muốn thay lựa chọn, chỉ dựa trên validation và chốt trước bước test. Để nhận Grad-CAM chọn một mô hình có nhánh CNN; mặc định ưu tiên Adaptive nếu đã chạy. Dự đoán ảnh mới trích CLS từ chính ảnh đó, không tra feature array train/validation. Đường dẫn ví dụ lấy một ảnh validation để kiểm tra pipeline, **không phải ca lâm sàng mới**. Chỉnh `NEW_IMAGE_PATH` nếu muốn dùng tệp khác.

### Phạm vi của dự đoán ảnh đơn
**Đầu vào:** đường dẫn ảnh mới và model đã lưu. **Đầu ra:** ba scores, nhãn có score lớn nhất, hash ảnh và metadata mô hình trong `new_image_result.json`.

Adaptive AACNN–ViT được chỉ định là mô hình chính và bắt buộc có trong danh sách train. Đây là lựa chọn cấu hình, không phải khẳng định model đó tốt nhất. Muốn thay lựa chọn phải dựa vào mục tiêu/validation trước khi xem test. Ảnh ví dụ lấy từ validation để thử luồng, không được mô tả là mẫu hoàn toàn chưa từng tham gia chọn mô hình.

In [ ]:
SELECTED_MODEL = "AACNN_Adaptive"
assert SELECTED_MODEL in MODELS_TO_RUN, "Phải train mô hình chính Adaptive trong run này."
NEW_IMAGE_PATH = BASE_DIR / val_rows[0]["relative_path"]
selected_dir = RUN_DIR / SELECTED_MODEL
selected_model = tf.keras.models.load_model(selected_dir / "model.keras", compile=False, safe_mode=True)
mode = MODES[SELECTED_MODEL]

def inputs_for_image(path, selected_mode):
    """Tạo tensor ảnh/CLS theo cùng hợp đồng train và cache."""
    image = cnn_image(path)[None, ...]
    if selected_mode == "cnn": return image
    cls = vit_batch([path]); release_vit()
    return cls if selected_mode == "vit" else (image, cls)

new_inputs = inputs_for_image(NEW_IMAGE_PATH, mode)
new_scores = np.asarray(selected_model(new_inputs, training=False))[0]
new_result = {"image_sha256": sha256_file(NEW_IMAGE_PATH), "model": SELECTED_MODEL,
              "class_order": CLASS_NAMES, "scores": new_scores.tolist(),
              "label": CLASS_NAMES[int(np.argmax(new_scores))], "mode": "real",
              "execution_source": "colab", "smoke": SMOKE_TEST}
write_json(selected_dir / "new_image_result.json", new_result)
print(new_result)

## 11 Đánh giá test một lần trong luồng chạy tất cả
Mặc định tự chạy test sau huấn luyện theo `RUN_TEST_ONCE=True`. Mô hình đánh giá chính được cố định là Adaptive AACNN–ViT trước khi xem test. Đây không phải chọn best bằng test. Nếu chỉ muốn phát triển/tune trên validation, đặt `RUN_TEST_ONCE=False` ở cấu hình đầu. Marker ngăn vô tình chạy lại cùng run; không sửa/xóa marker để dò test. Nếu thay cấu hình theo test, tập đó không còn là test độc lập cho kết luận mới. Nghiên cứu cố định 70/15/15. Smoke tự bỏ qua test chính thức; bản mặc định chạy đủ dữ liệu hợp lệ.

### Cách diễn giải đánh giá cuối
Cell này dùng đúng selected model đã xác định ở bước trước. Kết quả test có cùng định dạng validation nhưng được lưu tiền tố `test_`. Marker `test_started.json` lưu lựa chọn trước khi đánh giá và ngăn chạy cell lặp trong cùng run.

Việc có một tập test không loại trừ rò rỉ bệnh nhân nếu thiếu patient-level split. Nếu xem test rồi sửa kiến trúc hoặc chọn cấu hình để tăng test, cần công bố rằng đã dùng thông tin test trong phát triển và không coi nó là đánh giá độc lập cuối cùng.

In [ ]:
# Chạy test theo cờ cấu hình đầu notebook; không dùng kết quả để sửa/chọn mô hình.
if RUN_TEST_ONCE and not SMOKE_TEST:
    assert not SMOKE_TEST, "Smoke không phải thí nghiệm đánh giá cuối."
    assert SPLIT_MODE == "holdout" and test_rows, "Run này không có test độc lập."
    marker = RUN_DIR / "test_started.json"
    if marker.exists(): raise RuntimeError("Test đã bắt đầu trong run này; kiểm tra hồ sơ thay vì chạy lặp.")
    write_json(marker, {"model": SELECTED_MODEL, "time": datetime.now(timezone.utc).isoformat(),
                        "split_hash": SPLIT_HASH})
    features["test"] = extract_features(test_rows); release_vit()
    seq = DataSequence(test_rows, features["test"], mode)
    evaluate_and_save(selected_model, seq, test_rows, selected_dir, "test")
else:
    print("Test NOT_RUN — đang smoke hoặc RUN_TEST_ONCE=False.")

## 12 Grad-CAM thật của nhánh CNN
Gradient lấy theo score lớp mục tiêu qua activation convolution. Với fusion, **CLS ViT được giữ cố định**: bản đồ giải thích đường CNN của mô hình kết hợp, không giải thích đầy đủ ViT. Không gọi heatmap là ranh giới u, mask segmentation hoặc vị trí tổn thương đã xác nhận.

Dùng score softmax như nguồn mô hình; saturation có thể làm gradient yếu. Khi gradient mất kết nối hoặc CAM bằng 0, báo rõ trạng thái thay vì dựng heatmap giả. Không đổi sang phương pháp khác nhưng vẫn gọi cùng tên mà không ghi nhận.

### Giải thích đầu vào và đầu ra Grad-CAM
**Đầu vào:** tensor ảnh/CLS, model đã train và lớp mục tiêu (mặc định lớp dự đoán). Gradient đo độ nhạy của score mục tiêu với activation CNN. Trung bình gradient theo không gian tạo trọng số cho các kênh; tổng có trọng số và ReLU tạo bản đồ, sau đó chuẩn hóa để hiển thị.

**Đầu ra:** `gradcam_raw.npy`, `gradcam_overlay.png`, `gradcam_status.json`. Trạng thái PASS nghĩa là tạo được bản đồ hữu hạn khác 0, chưa chứng minh bản đồ định vị đúng tổn thương. ZERO_CAM là bản đồ bằng 0; UNAVAILABLE áp dụng cho ViT Head không có nhánh CNN. Lỗi gradient không được che bằng ảnh màu giả.

Màu nóng biểu thị mức đóng góp tương đối theo cách chuẩn hóa trong ảnh đó. Không so sánh tuyệt đối màu giữa hai ảnh như cùng một thang nguy cơ. Heatmap chỉ là công cụ giải thích hỗ trợ, không thay kết luận bác sĩ.

In [ ]:
def make_gradcam(model, inputs, class_index=None):
    """Grad-CAM CNN; trả bản đồ chuẩn hóa và chẩn đoán gradient."""
    convs = [layer for layer in model.layers if isinstance(layer, layers.Conv2D)]
    if not convs: raise ValueError("Model không có convolution; ViT_Head không hỗ trợ CAM CNN.")
    target = model.get_layer("conv5_block3_out") if any(l.name == "conv5_block3_out" for l in model.layers) else convs[-1]
    grad_model = tf.keras.Model(model.inputs, [target.output, model.output])
    tensor_inputs = tf.nest.map_structure(tf.convert_to_tensor, inputs)
    with tf.GradientTape() as tape:
        # Theo dõi input để gradient vẫn được ghi khi backbone bị đóng băng.
        for value in tf.nest.flatten(tensor_inputs): tape.watch(value)
        activations, scores = grad_model(tensor_inputs, training=False)
        index = int(tf.argmax(scores[0])) if class_index is None else int(class_index)
        assert 0 <= index < NUM_CLASSES
        objective = scores[:, index]
    gradients = tape.gradient(objective, activations)
    if gradients is None: raise RuntimeError("Gradient không liên thông tới lớp CNN được chọn.")
    if not bool(tf.reduce_all(tf.math.is_finite(gradients))): raise RuntimeError("Gradient chứa NaN/Inf.")
    weights = tf.reduce_mean(gradients, axis=(1,2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(activations * weights, axis=-1))[0]
    maximum = float(tf.reduce_max(cam))
    if maximum <= 1e-12:
        return np.zeros(cam.shape, np.float32), {"status": "ZERO_CAM", "target_layer": target.name, "target_class": CLASS_NAMES[index]}
    return (cam / maximum).numpy(), {"status": "PASS", "target_layer": target.name,
                                   "target_class": CLASS_NAMES[index], "scope": "CNN branch; fixed ViT CLS"}

if mode == "vit":
    cam_info = {"status": "UNAVAILABLE", "reason": "ViT_Head không có CNN"}
else:
    cam, cam_info = make_gradcam(selected_model, new_inputs)
    np.save(selected_dir / "gradcam_raw.npy", cam, allow_pickle=False)
    original = rgb_image(NEW_IMAGE_PATH)
    heat = tf.image.resize(cam[...,None], [original.height, original.width]).numpy()[...,0]
    fig, axes = plt.subplots(1,2,figsize=(12,5))
    axes[0].imshow(original); axes[0].set_title("Ảnh gốc")
    axes[1].imshow(original)
    if cam_info["status"] == "PASS": axes[1].imshow(heat, cmap="jet", alpha=0.4, vmin=0, vmax=1)
    axes[1].set_title("Grad-CAM nhánh CNN — " + cam_info["status"])
    for ax in axes: ax.axis("off")
    fig.tight_layout(); fig.savefig(selected_dir / "gradcam_overlay.png", dpi=160); plt.show(); plt.close(fig)
write_json(selected_dir / "gradcam_status.json", cam_info)
print(cam_info)

## 13 Kiểm tra nạp trọng số và benchmark Colab
Nạp checkpoint vào builder mới, so sánh cùng batch. Việc này độc lập với kiểm tra `.keras` trước đó. Số đo latency dưới đây chỉ phản ánh TensorFlow forward với input đã chuẩn bị trên runtime hiện tại; **không gồm decode/ViT/Grad-CAM và không phải hiệu năng điện thoại**. Chỉ chạy 10 mẫu lặp, không dùng làm benchmark lâm sàng hoặc tải đồng thời.

### Những gì đã và chưa được đo
**Đầu vào:** model, trọng số và cùng một ảnh đã chuẩn bị tensor. **Đầu ra:** kiểm tra nạp weights, số lần đo, latency thô, p50/p95 và manifest artifact. p50 là trung vị; p95 là phân vị 95 từ số mẫu thực hiện, ở đây chỉ có 10 lần lặp nên độ ổn định thống kê hạn chế.

Artifact gồm model cùng cấu hình cần thiết, không chỉ file trọng số. Bản `.keras` fusion không chứa pretrained ViT PyTorch; muốn dự đoán ảnh mới vẫn cần đúng ViT/processor. `mobile_export_status=NOT_IMPLEMENTED` phải được giữ cho tới khi có chuyển đổi, parity và kiểm thử thiết bị thật.

In [ ]:
rebuilt = BUILDERS[SELECTED_MODEL]()
rebuilt.load_weights(selected_dir / "best.weights.h5")
reloaded_scores = np.asarray(rebuilt(new_inputs, training=False))[0]
np.testing.assert_allclose(new_scores, reloaded_scores, atol=1e-5, rtol=1e-4)
write_json(selected_dir / "weights_roundtrip.json", {"status": "PASS", "max_abs_error": float(np.max(np.abs(new_scores-reloaded_scores)))})
del rebuilt; gc.collect()
for _ in range(2): np.asarray(selected_model(new_inputs, training=False))
latencies = []
for _ in range(10):
    start = time.perf_counter()
    np.asarray(selected_model(new_inputs, training=False))  # Đọc ra CPU để chờ phép tính hoàn tất.
    latencies.append((time.perf_counter()-start)*1000)
write_json(selected_dir / "benchmark_colab_forward.json", {
    "scope": "warm TF forward only; excludes decode, ViT extraction, CAM", "samples": len(latencies),
    "milliseconds": latencies, "p50_ms": float(np.percentile(latencies,50)),
    "p95_ms": float(np.percentile(latencies,95)), "environment": ENVIRONMENT,
    "keras_bytes": (selected_dir/"model.keras").stat().st_size})
manifest = {"run_id": RUN_ID, "model": SELECTED_MODEL, "smoke": SMOKE_TEST,
    "class_order": CLASS_NAMES, "cnn_input": [*IMG_SIZE,3], "cnn_range": [0,1],
    "vit_size": VIT_SIZE, "vit_feature_dim": FEATURE_DIM, "vit_model": VIT_MODEL_ID,
    "vit_revision": RESOLVED_VIT_REVISION, "processor_file": "../vit_processor.json",
    "preprocessing_version": PREPROCESS_VERSION, "split_manifest_sha256": SPLIT_HASH,
    "keras_sha256": sha256_file(selected_dir/"model.keras"),
    "weights_sha256": sha256_file(selected_dir/"best.weights.h5"),
    "gradcam": cam_info, "mobile_export_status": "NOT_IMPLEMENTED",
    "test_status": "SEE_TEST_FILES" if (selected_dir/"test_metrics.json").exists() else "NOT_RUN",
    "limitations": ["image-level split; patient independence unverified", "not clinical validation",
                     "two frameworks; no mobile runtime integration", "source license not verified",
                     "unequal training budgets/pretraining across models"]}
write_json(selected_dir / "artifact_manifest.json", manifest)
print("Đã ghi manifest:", selected_dir / "artifact_manifest.json")

## 14 Hồ sơ thực nghiệm và công việc tiếp theo

Mỗi run lưu cấu hình, môi trường, dữ liệu/split manifest, log huấn luyện, best weights, full model đã thử nạp lại, dự đoán, metrics và hình. Thư mục output trên Drive phải được giữ cùng bản notebook đã chạy. Không đưa dataset, weights, bí mật hoặc hồ sơ thật vào Git.

- **TODO [SRC-01]**: Xác minh bài báo/repository tác giả và license dataset, pretrained weights và code; ghi source register cho hai báo cáo.
- **TODO [DATA-01]**: Có patient IDs đáng tin trước chia theo bệnh nhân; kiểm tra near-duplicate có protocol trước khi train, không chỉnh split theo test.
- **TODO [EXP-01]**: So sánh augmentation đồng bộ, class weights/loss và fine-tuning từng thay đổi trên validation; giữ protocol và run ID.
- **TODO [EXP-02]**: Nếu so sánh kiến trúc, thiết kế ngân sách/khởi tạo/pretrained công bằng; chạy nhiều seed nếu tài nguyên cho phép, ghi mean/std từ run thực.
- **TODO [MOB-01]**: Làm thử export cả ResNet/fusion và ViT cùng preprocessing; chọn runtime sau khi kiểm tra toán tử, RAM, parity và Grad-CAM.
- **TODO [MOB-02]**: Đo trên thiết bị Android/iOS; không lấy số đo Colab thay điện thoại.
- **TODO [THS-01]**: Sau mỗi lệnh Antigravity, cập nhật trực tiếp báo cáo Android và iOS, nguồn và evidence; Git commit code/config/tài liệu rồi báo chat file sửa, mục báo cáo, test và commit.

### Ghi nguồn đúng mức đã biết
Tệp nguồn là bản nhóm lưu làm đầu vào, không chứng minh tác giả notebook là tác giả bài báo. Các URL sau là điểm tra cứu chính thức để đối chiếu khi chạy, **chưa được xác minh nội dung trong lần biên soạn này**:
- TensorFlow/Keras: https://www.tensorflow.org/api_docs/python/tf/keras
- Hugging Face ViT: https://huggingface.co/docs/transformers/model_doc/vit
- Model ID dùng trong code: https://huggingface.co/google/vit-base-patch16-224-in21k
- scikit-learn metrics: https://scikit-learn.org/stable/modules/model_evaluation.html

### Trạng thái kiểm chứng của phiên bản này
Đã kiểm tra cấu trúc notebook, cú pháp cell và các hàm dữ liệu độc lập bằng bộ dữ liệu tổng hợp nhỏ. **Chưa chạy toàn bộ huấn luyện trên dữ liệu thật, tải ViT pretrained hoặc kiểm thử điện thoại** trong lúc tạo tệp; các bước này cần chạy trong môi trường Colab để xác minh. Kiểm tra kỹ thuật cục bộ dùng TensorFlow CPU 2.18.1, ảnh tổng hợp 32×32, feature tổng hợp và ResNet không tải pretrained; không phải số liệu độ chính xác của nghiên cứu. Không có accuracy/F1/latency mới được khẳng định trước khi chạy.

## 15 Mẫu ghi nhận sau khi chạy

Điền từ tệp đầu ra thực tế; không điền theo kỳ vọng hoặc kết quả lịch sử.

| Nội dung | Giá trị cần ghi |
|---|---|
| Run ID và loại run | Tên thư mục; smoke hay experiment |
| Môi trường | GPU, framework, thời điểm |
| Dữ liệu | Nguồn/license, số ảnh từng lớp sau làm sạch, số loại trùng |
| Chia tập | Số mẫu train/validation/test; split hash; giới hạn patient ID |
| Thay đổi so với nguồn | Dedup, split, loss, augmentation, BN, chọn checkpoint |
| Mô hình | Tên, tham số, pretrained revision, cấu hình train |
| Validation | Accuracy, macro F1, recall/F1 từng lớp |
| Test | NOT_RUN hoặc số đo cuối; nêu model đã chốt trước test |
| Lưu và nạp | PASS/FAIL cùng sai lệch output |
| Grad-CAM | PASS/ZERO_CAM/UNAVAILABLE/FAIL; ảnh thử và phạm vi giải thích |
| Hiệu năng | Đúng phạm vi đo; không quy số đo Colab thành điện thoại |
| Hạn chế và công việc tiếp theo | Điều chưa kiểm chứng và cách xử lý |

Cách viết kết quả: “Trong run [ID], mô hình [tên] đạt [số đo] trên [tập và số mẫu], theo split [hash]. Kết quả chỉ phản ánh protocol và dữ liệu được mô tả; [hạn chế cụ thể].” Chỉ điền số khi có file kết quả kiểm tra được.

Để giải thích lại được toàn bộ quy trình, người thực hiện cần trả lời: Vì sao chia ba tập? Vì sao ảnh và CLS phải đúng cùng mẫu? Vì sao checkpoint chọn bằng validation? Vì sao scores không tự là xác suất bệnh? Vì sao Grad-CAM này chỉ giải thích nhánh CNN? Vì sao chưa thể tuyên bố chạy được trên điện thoại?


## 16 Đóng gói mô hình chính đầy đủ để tải về

Bước này không huấn luyện lại hoặc thay trọng số. Chúng tôi gom checkpoint tốt nhất, mô hình Keras, pretrained ViT đúng revision, processor và metadata của run vào một gói ZIP trên Drive. Không đóng gói dataset hoặc token. Gói có thể lớn vì chứa đủ hai nhánh.

**Đầu vào:** run train chính thức hoàn thành, Adaptive đã nạp lại khớp, kết quả validation và trạng thái Grad-CAM. **Đầu ra:** `LungCT_Primary_<RUN_ID>.zip`, SHA256 và manifest có danh sách file.

Trạng thái Grad-CAM khác PASS làm bước đóng gói dừng để xử lý; không bỏ âm thầm chức năng cần cho mô hình chính. Smoke không được đóng gói thành mô hình chính. Kiểm tra checksum sau tải. Gói này hỗ trợ tái tạo pipeline máy tính và chuẩn bị chuyển đổi, chưa xác nhận tương thích hoặc hiệu năng trên điện thoại. Sau chạy tất cả, tải ZIP từ thư mục run trên Drive.


Gói còn có `predict.py` và `custom_layers.py`, dữ liệu dự đoán thô và hình kết quả. Một tiến trình CPU mới nạp cả hai nhánh từ file cục bộ với chế độ Hugging Face offline, rồi so output ảnh thử với notebook. Kiểm tra này không chứng minh mobile-ready, cũng không thay parity trên nhiều ảnh hoặc môi trường thật.


In [ ]:
import shutil, zipfile
assert not SMOKE_TEST, "Không xuất run thử thành mô hình chính."
assert SELECTED_MODEL == "AACNN_Adaptive"
assert cam_info.get("status") == "PASS", "Grad-CAM chưa đạt trên ảnh thử; kiểm tra trước xuất mô hình chính."
for filename in ("roundtrip.json", "weights_roundtrip.json"):
    assert json.loads((selected_dir / filename).read_text())["status"] == "PASS"
bundle_dir = RUN_DIR / "primary_bundle"
if bundle_dir.exists(): raise FileExistsError("Gói đã tồn tại; kiểm tra thay vì ghi đè.")
(bundle_dir / "fusion").mkdir(parents=True)
for filename in ("model.keras", "best.weights.h5", "artifact_manifest.json",
                 "roundtrip.json", "weights_roundtrip.json", "training.json", "summary.txt",
                 "validation_metrics.json", "gradcam_status.json"):
    shutil.copy2(selected_dir / filename, bundle_dir / "fusion" / filename)
# Giữ cả dữ liệu gốc và hình để hội đồng có thể đối chiếu số liệu.
for filename in ("test_metrics.json", "test_confusion_matrix.json", "test_predictions.csv",
                 "validation_confusion_matrix.json", "validation_predictions.csv", "history.json",
                 "test_cm.png", "test_roc.png", "validation_cm.png", "validation_roc.png", "history.png",
                 "gradcam_raw.npy", "gradcam_overlay.png", "new_image_result.json", "benchmark_colab_forward.json"):
    if (selected_dir / filename).exists():
        shutil.copy2(selected_dir / filename, bundle_dir / "fusion" / filename)
for filename in ("config.json", "environment.json", "pip-freeze.txt", "vit_processor.json",
                 "vit_source.json", "split_manifest.json", "split_summary.json", "class_weights.json", "preflight.json", "augmentation_summary.json", "training_samples_manifest.json"):
    shutil.copy2(RUN_DIR / filename, bundle_dir / filename)
# Gói phải có cả ViT, không chỉ classifier fusion nhận vector CLS.
vit_for_export = get_vit().to("cpu")
vit_for_export.save_pretrained(bundle_dir / "vit", safe_serialization=True)
processor.save_pretrained(bundle_dir / "vit")
release_vit()
del vit_for_export; gc.collect()
write_custom = 'import tensorflow as tf\nfrom tensorflow.keras import layers, regularizers\n\n@tf.keras.utils.register_keras_serializable(package="LungCT")\nclass ResNetPreprocess(layers.Layer):\n    """Chuyển RGB [0,1] sang preprocessing ResNet ImageNet đúng một lần."""\n    def call(self, x): return tf.keras.applications.resnet.preprocess_input(x * 255.0)\n\n@tf.keras.utils.register_keras_serializable(package="LungCT")\nclass StackTokens(layers.Layer):\n    def call(self, inputs): return tf.stack(inputs, axis=1)\n    def compute_output_shape(self, shapes): return (shapes[0][0], len(shapes), shapes[0][-1])\n\n@tf.keras.utils.register_keras_serializable(package="LungCT")\nclass AdaptiveGate(layers.Layer):\n    """Học alpha trong [0,1] để trộn hai token có cùng chiều."""\n    def __init__(self, d_hidden=128, dropout=0.25, l2=1e-4, **kwargs):\n        super().__init__(**kwargs)\n        self.d_hidden, self.dropout_rate, self.l2_rate = d_hidden, dropout, l2\n        self.concat = layers.Concatenate()\n        self.d1 = layers.Dense(d_hidden, activation="relu", kernel_regularizer=regularizers.l2(l2))\n        self.drop = layers.Dropout(dropout)\n        self.logit = layers.Dense(1, kernel_initializer="zeros", bias_initializer="zeros")\n    def build(self, input_shape):\n        # Tạo biến rõ ràng để Keras khôi phục sublayer weights an toàn.\n        a, b = input_shape\n        if a[-1] != b[-1]: raise ValueError("Hai token phải cùng chiều.")\n        self.concat.build(input_shape)\n        self.d1.build((a[0], a[-1] + b[-1]))\n        self.drop.build((a[0], self.d_hidden))\n        self.logit.build((a[0], self.d_hidden))\n        super().build(input_shape)\n    def call(self, inputs, training=None):\n        a, b = inputs\n        alpha = tf.sigmoid(self.logit(self.drop(self.d1(self.concat([a, b])), training=training)))\n        return alpha * a + (1-alpha) * b\n    def get_config(self):\n        return {**super().get_config(), "d_hidden": self.d_hidden,\n                "dropout": self.dropout_rate, "l2": self.l2_rate}\n\n'
(bundle_dir / "custom_layers.py").write_text(write_custom, encoding="utf-8")
PREDICTOR_SOURCE = '"""Inference CPU offline: python predict.py --image PATH --output result.json."""\nimport os\nos.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")\nos.environ.setdefault("HF_HUB_OFFLINE", "1")\nos.environ.setdefault("TRANSFORMERS_OFFLINE", "1")\nimport argparse, json, hashlib\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport tensorflow as tf\nimport torch\nfrom transformers import ViTModel, ViTImageProcessor\nimport custom_layers  # Registers the serialized LungCT classes.\n\ndef predict(image_path):\n    """Load both branches locally; preserve resize, normalization and class order."""\n    root = Path(__file__).resolve().parent\n    config = json.loads((root / "config.json").read_text(encoding="utf-8"))\n    with Image.open(image_path) as original:\n        rgb = original.convert("RGB").copy()\n    cnn = np.asarray(rgb.resize(tuple(config["IMG_SIZE"]), Image.Resampling.NEAREST), dtype=np.float32)[None] / 255.0\n    processor = ViTImageProcessor.from_pretrained(root / "vit", local_files_only=True)\n    vit = ViTModel.from_pretrained(root / "vit", local_files_only=True).float().eval()\n    resized = rgb.resize(tuple(config["VIT_SIZE"]), Image.Resampling.NEAREST)\n    batch = processor(images=[resized], return_tensors="pt")\n    with torch.inference_mode():\n        cls = vit(**batch).last_hidden_state[:,0,:].numpy().astype(np.float32)\n    model = tf.keras.models.load_model(root / "fusion" / "model.keras", compile=False, safe_mode=True)\n    scores = np.asarray(model((cnn, cls), training=False))[0]\n    assert np.isfinite(scores).all() and np.allclose(scores.sum(), 1, atol=1e-4)\n    names = config["CLASS_NAMES"]\n    return {"class_order": names, "scores": scores.tolist(), "label": names[int(scores.argmax())],\n            "execution_source": "computer_cpu_offline", "mobile_ready": False}\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--image", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args()\n    result = predict(args.image)\n    Path(args.output).write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(json.dumps(result, ensure_ascii=False))\n'
readme = """LungCT Primary model source bundle
Primary model: Adaptive AACNN-ViT. This is NOT a mobile-ready runtime artifact.
Fusion: fusion/model.keras and fusion/best.weights.h5.
ViT: vit/ with pretrained weights, config and processor. Load from local folder after download.
Class order: Benign, Malignant, Normal.
CNN input: RGB resized to 512x512 nearest, float32 [0,1]; model contains ResNet preprocessing.
ViT input: RGB resized to 224x224 nearest, then saved ViT processor; CLS at index 0.
Custom Keras classes: ResNetPreprocess, StackTokens, AdaptiveGate (definitions in custom_layers.py; import this module before loading).
Load Keras with compile=False and safe_mode=True after registering these classes.
Use the environment and metadata files for reproducibility.
Run offline on a computer: python predict.py --image /path/to/image.png --output result.json
The package includes one-image offline parity evidence; validate more representative images before deployment.
Grad-CAM in this notebook explains the CNN branch while the ViT CLS is fixed.
Android/iOS conversion, parity, offline runtime and offline Grad-CAM are still NOT VERIFIED.
Review source/model/data licenses before distribution.
"""
(bundle_dir / "README.txt").write_text(readme, encoding="utf-8")
(bundle_dir / "HUONG_DAN_SU_DUNG.txt").write_text('# Hướng dẫn nhận và sử dụng gói mô hình chính LungCT\n\n## File tải về sau khi chạy thành công\nTải LungCT_Primary_<RUN_ID>.zip và file .zip.sha256 nằm cùng thư mục run trên Google Drive.\nKhông tải file .ipynb rồi đặt vào app. Notebook là mã chạy thí nghiệm, không phải mô hình.\nKhông chỉ lấy best.weights.h5 hoặc model.keras vì pipeline còn cần nhánh ViT và processor.\n\n## Vai trò từng file\nfusion/model.keras: kiến trúc và trọng số CNN/fusion Keras, nhận ảnh CNN cùng vector CLS ViT.\nfusion/best.weights.h5: trọng số CNN/fusion tốt nhất theo validation macro F1; không chứa toàn pipeline ViT.\nvit/: trọng số pretrained ViT, config và processor, cần để tính CLS cho ảnh mới.\ncustom_layers.py: đăng ký các lớp tùy chỉnh trước khi nạp model.keras.\npredict.py: chương trình dự đoán trên máy tính CPU bằng file cục bộ; không phải runtime React Native.\nconfig.json, vit_processor.json, vit_source.json: đầu vào, thứ tự lớp, tiền xử lý và nguồn pretrained.\nbundle_manifest.json: danh sách file, kích thước và checksum; không phải trọng số mô hình.\noffline_parity.json: kết quả so sánh một ảnh giữa notebook và tiến trình CPU độc lập.\nfusion/*metrics.json, *predictions.csv, *confusion_matrix.json và ảnh PNG: bằng chứng nghiên cứu; không phải file chạy AI.\n\n## Thử gói độc lập trên máy tính\n1. Giải nén ZIP vào một thư mục riêng, giữ nguyên cấu trúc.\n2. Tạo môi trường Python phù hợp với environment.json và pip-freeze.txt. Không cài toàn bộ gói Colab tùy tiện vào dự án app; pip-freeze là bản ghi môi trường, cần chọn dependency cần thiết và kiểm tra tương thích.\n3. Chạy: python predict.py --image DUONG_DAN_ANH.png --output ket_qua.json\n4. Kiểm tra class_order là Benign, Malignant, Normal; scores hữu hạn, tổng gần 1.\n5. Dùng cùng ảnh và so với output notebook trong dung sai đã quy định. Kiểm thử nhiều ảnh đại diện trước triển khai.\n\n## Trạng thái gắn vào điện thoại\nGói ZIP này CHƯA phải artifact mobile-ready. Keras và PyTorch không tự chạy trong React Native chỉ bằng sao chép file.\nBước tiếp theo: chọn bộ thực thi sau làm thử; chuyển cả hai nhánh và preprocessing hoặc thiết kế phương án đã được chấp thuận; kiểm tra output trước/sau; triển khai Grad-CAM thật; đo thời gian/RAM/kích thước trên Android và iOS.\nĐịnh dạng đích chưa chốt. Không tự gọi file ZIP, .keras hoặc .h5 là mô hình Android/iOS đã nghiệm thu.\nCác vị trí đích dự kiến là mobile/assets/models/android/ và mobile/assets/models/ios/ sau khi tạo artifact tương thích.\nKhi bàn giao mobile phải có manifest chỉ rõ file nào nạp bằng runtime nào, code inference, class order, preprocessing, Grad-CAM, checksum và bằng chứng tắt mạng.\nNếu thiếu một nhánh hoặc runtime không tạo được Grad-CAM, phải báo rõ và giải quyết trước xác nhận đạt.\n', encoding="utf-8")
(bundle_dir / "predict.py").write_text(PREDICTOR_SOURCE, encoding="utf-8")
# Chạy tiến trình mới với HF offline; predictor không phụ thuộc biến hoặc feature cache trong notebook.
offline_output = selected_dir / "offline_prediction.json"
offline_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "-1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
               "TF_NUM_INTRAOP_THREADS": "2", "TF_NUM_INTEROP_THREADS": "2"}
check = subprocess.run([sys.executable, str(bundle_dir / "predict.py"), "--image", str(NEW_IMAGE_PATH),
                        "--output", str(offline_output)], env=offline_env, capture_output=True, text=True)
(selected_dir / "offline_predict_stdout.txt").write_text(check.stdout, encoding="utf-8")
(selected_dir / "offline_predict_stderr.txt").write_text(check.stderr, encoding="utf-8")
if check.returncode != 0:
    raise RuntimeError("Gói độc lập chưa chạy được; xem offline_predict_stderr.txt. Chưa xuất ZIP hoàn tất.")
independent = json.loads(offline_output.read_text())
np.testing.assert_allclose(new_scores, independent["scores"], atol=1e-4, rtol=1e-3)
write_json(bundle_dir / "offline_parity.json", {"status": "PASS", "scope": "one image; fresh CPU process, local files only",
    "max_abs_error": float(np.max(np.abs(new_scores-np.asarray(independent["scores"]))))})
shutil.copy2(offline_output, bundle_dir / "offline_prediction.json")
files = {str(path.relative_to(bundle_dir)): {"sha256": sha256_file(path), "bytes": path.stat().st_size}
         for path in sorted(bundle_dir.rglob("*")) if path.is_file()}
write_json(bundle_dir / "bundle_manifest.json", {"run_id": RUN_ID, "primary_model": SELECTED_MODEL,
    "vit_revision": RESOLVED_VIT_REVISION, "files": files, "mobile_ready": False,
    "gradcam_scope": "CNN branch with fixed ViT CLS", "dataset_included": False})
zip_path = RUN_DIR / f"LungCT_Primary_{RUN_ID}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as archive:
    for path in sorted(bundle_dir.rglob("*")):
        if path.is_file(): archive.write(path, arcname=str(path.relative_to(bundle_dir)))
checksum = sha256_file(zip_path)
zip_path.with_suffix(".zip.sha256").write_text(checksum + "  " + zip_path.name + "\n")
print("FILE CẦN TẢI: gói mô hình chính đầy đủ để chuyển đổi/tích hợp, chưa mobile-ready:", zip_path)
print("TẢI KÈM để đối chiếu checksum:", zip_path.with_suffix(".zip.sha256"))
print("Giải nén và đọc HUONG_DAN_SU_DUNG.txt trước khi dùng. Không chỉ lấy file .h5 hoặc .keras.")
print("Dung lượng MiB:", round(zip_path.stat().st_size / 1024**2, 2))
print("SHA256:", checksum)


## 17 Điều kiện sử dụng kết quả để báo cáo hội đồng

| Nội dung cần chứng minh | Trạng thái trước khi chạy trên dữ liệu thật | Bằng chứng cần lưu sau chạy |
|---|---|---|
| Chia 70/15/15 sau loại trùng | Đã có logic; số lượng thực chờ dữ liệu | split_manifest, split_summary, kiểm tra giao hash |
| Train đủ protocol và chọn best validation | Đã có code, chưa có kết quả nghiên cứu | history, checkpoint, training config |
| Kiến trúc lưu/nạp được | Kiểm tra nhỏ cục bộ đã thực hiện; preflight chạy lại ở Colab | preflight, roundtrip, weights_roundtrip |
| Chất lượng ba lớp | Chưa xác minh | test metrics, predictions, confusion matrix; recall từng lớp |
| Grad-CAM | Đã thử gradient với model tổng hợp; ảnh thật chưa xác minh | CAM và metadata trên các ảnh đại diện, cả ca lỗi |
| Gói độc lập trên máy tính | Đã thêm kiểm tra tiến trình mới, chưa chạy với pretrained thật | offline_parity và kiểm tra tái chạy ngoài notebook |
| Android/iOS ngoại tuyến | Chưa triển khai hoặc kiểm thử | artifact đã chuyển đổi, parity, RAM/latency và video thiết bị |
| Nguồn và license | Cần đối chiếu tác giả/dataset/pretrained | nguồn gốc đã xác minh và danh mục trích dẫn |
| Tính độc lập bệnh nhân | Chưa có patient mapping đáng tin | patient-level split hoặc công bố hạn chế đúng mức |

**Chỉ sau khi chạy và xem các bằng chứng mới có thể xác nhận kết quả thí nghiệm.** Không tồn tại một cờ “chuẩn hội đồng” thay thế tiêu chí của giảng viên/khoa. Có thể trình bày đây là thực nghiệm nghiên cứu theo protocol đã nêu; không suy rộng thành hệ thống chẩn đoán đã kiểm định lâm sàng.

Nguồn kỹ thuật đã đối chiếu khi rà soát: [Keras Grad-CAM](https://keras.io/examples/vision/grad_cam/), [Keras serialization](https://keras.io/guides/serialization_and_saving/), [Keras losses](https://keras.io/api/losses/). Các nguồn này hỗ trợ cách triển khai kỹ thuật, không chứng minh kết quả mô hình hoặc nguồn tác giả của notebook đầu vào.

Hạn chế vận hành còn lại: chưa có kiểm chứng resume đầy đủ qua Colab mất phiên; best checkpoint và history được lưu nhưng không đồng nghĩa khôi phục chính xác optimizer/epoch. Ghi run bị gián đoạn là incomplete. Không chạy lại từ đầu rồi gộp lịch sử thành một run liên tục nếu chưa ghi rõ.


## 18 File cần tải và hướng dẫn bàn giao cho người tích hợp app

### A File tải về ngay sau lượt train này
**Tải `LungCT_Primary_<RUN_ID>.zip` và `LungCT_Primary_<RUN_ID>.zip.sha256`** ở đường dẫn được cell xuất gói in ra. Trên Colab, mở biểu tượng thư mục, vào Drive và đúng thư mục run, hoặc mở thư mục đó trong Google Drive để tải ZIP. Chỉ có ZIP hoàn tất sau khi các kiểm tra bắt buộc qua; nếu cell dừng thì chưa có gói đạt điều kiện của notebook.

| File hoặc thư mục | Vai trò | Có thể bỏ khi bàn giao mô hình không |
|---|---|---|
| `fusion/model.keras` | Model CNN/fusion, nhận ảnh và CLS | Không |
| `fusion/best.weights.h5` | Checkpoint tốt nhất để khôi phục/chuyển đổi | Giữ trong bản bàn giao nguồn |
| `vit/` | Model ViT và processor để tính CLS ảnh mới | Không |
| `custom_layers.py` | Lớp tùy chỉnh để nạp Keras | Không khi nạp theo pipeline này |
| `predict.py` | Thử dự đoán độc lập trên máy tính | Giữ để kiểm tra pipeline |
| `config.json` và manifest/processor | Hợp đồng đầu vào, class order, revision và checksum | Không |
| `HUONG_DAN_SU_DUNG.txt` | Hướng dẫn bằng tiếng Việt kèm theo ZIP | Giữ |
| Metrics, predictions và hình | Số liệu và minh chứng cho báo cáo | Giữ trong hồ sơ nghiên cứu; không cần bundle vào bản cài |

### B Thử sau khi tải
Giải nén giữ cấu trúc → tạo môi trường Python từ các phiên bản cần thiết đã ghi → chạy `python predict.py --image DUONG_DAN_ANH.png --output ket_qua.json` → đối chiếu kết quả với notebook trên cùng ảnh. Chương trình nạp từ file cục bộ; không huấn luyện lại. `pip-freeze.txt` chứa nhiều gói Colab, không phải danh sách phải cài hết vào ứng dụng.

### C File cuối để chạy trực tiếp trong app
**Chưa có trong notebook này.** Cần bước xuất/chuyển đổi tương thích runtime Android/iOS đã lựa chọn, kèm code tiền xử lý/suy luận/Grad-CAM và kiểm tra parity. Chưa chốt runtime nên chưa được chỉ định đuôi file đích hoặc khẳng định chỉ cần `.tflite`, `.onnx` hay `.keras`.

Tên ZIP có chữ Primary chỉ có nghĩa đây là gói nguồn của mô hình chính trong thực nghiệm. Không đồng nghĩa đã đạt chất lượng cuối, chạy được điện thoại hoặc được nghiệm thu. Chỉ ghi mobile-ready khi có artifact, code tích hợp và minh chứng thiết bị thật. Giữ gói nguồn đã train để thực hiện bước đó, không đổi đuôi file để giả chuyển định dạng.


## 19 Tải ảnh lên để thử dự đoán và kiểm tra pipeline

Sau khi chạy train và lưu model, khối dưới hiển thị bộ chọn ảnh cùng nút **Dự đoán và tạo Grad-CAM**. Chọn một hoặc nhiều JPG/PNG, rồi nhấn nút. Khối này không chặn việc Chạy tất cả để chờ chọn ảnh; chỉ xử lý khi người thực hiện nhấn nút. Nếu runtime đã mất, phải nạp lại gói mô hình bằng hướng dẫn bên dưới hoặc train lại khi thực sự cần; không mặc định model còn trong RAM.

**Pipeline:** ảnh được tải → kiểm tra định dạng và kích thước → RGB → resize CNN/ViT theo đúng cấu hình → CLS của chính ảnh đó → Adaptive fusion → scores và nhãn → Grad-CAM nhánh CNN → hiển thị và lưu.

Ảnh tải lên chỉ là thử ảnh đơn theo yêu cầu người thực hiện. Không tự ghép vào 15% test hoặc dùng để tính accuracy khi chưa có nhãn chuẩn và protocol. Với ảnh chụp phim hoặc ảnh ngoài phân bố dataset, kết quả chưa chứng minh tính phù hợp lâm sàng.

Mỗi ảnh tạo một thư mục trong `RUN_DIR/uploaded_image_tests/`. Bên trong có ảnh đầu vào nguyên bản, `cnn_input.npy`, `vit_cls.npy`, `scores.npy`, `gradcam_raw.npy` khi có, ảnh overlay và `result.json`. Các mảng giúp đối chiếu từng bước giữa notebook và pipeline app sau này; không đưa tệp thử và ảnh bệnh nhân vào bản cài app.


In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display, clear_output

UPLOAD_MAX_BYTES = 20 * 1024 * 1024
UPLOAD_MAX_PIXELS = 25_000_000  # Giới hạn kỹ thuật thử ảnh, không phải tiêu chí chất lượng CT.

def predict_uploaded_bytes(content, original_name):
    """Chạy pipeline ảnh đơn và lưu tensor trung gian, scores, CAM và metadata."""
    raw = bytes(content)
    if len(raw) > UPLOAD_MAX_BYTES: raise ValueError("Ảnh vượt giới hạn 20 MiB của phần thử ảnh.")
    with Image.open(io.BytesIO(raw)) as probe:
        if probe.format not in {"JPEG", "PNG"}: raise ValueError("Chỉ nhận nội dung JPEG hoặc PNG thật.")
        if probe.width * probe.height > UPLOAD_MAX_PIXELS: raise ValueError("Ảnh vượt giới hạn số pixel.")
        probe.verify()
    image_hash = hashlib.sha256(raw).hexdigest()
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    dest = RUN_DIR / "uploaded_image_tests" / f"{stamp}_{image_hash[:12]}"
    dest.mkdir(parents=True, exist_ok=False)
    suffix = ".png" if raw.startswith(b"\x89PNG") else ".jpg"
    image_path = dest / ("input_original" + suffix)
    image_path.write_bytes(raw)
    start = time.perf_counter()
    image_tensor, cls_tensor = inputs_for_image(image_path, "fusion")
    scores = np.asarray(selected_model((image_tensor, cls_tensor), training=False))[0]
    if not np.isfinite(scores).all() or not np.allclose(scores.sum(), 1, atol=1e-4):
        raise RuntimeError("Scores không hợp lệ.")
    class_index = int(scores.argmax())
    np.save(dest / "cnn_input.npy", image_tensor, allow_pickle=False)
    np.save(dest / "vit_cls.npy", cls_tensor, allow_pickle=False)
    np.save(dest / "scores.npy", scores, allow_pickle=False)
    classification_seconds = time.perf_counter() - start
    cam_status = {"status": "FAIL"}
    try:
        cam, cam_status = make_gradcam(selected_model, (image_tensor, cls_tensor), class_index)
        np.save(dest / "gradcam_raw.npy", cam, allow_pickle=False)
    except Exception as error:
        cam_status = {"status": "FAIL", "error": repr(error)}
    result = {"original_filename": Path(original_name).name, "image_sha256": image_hash,
              "run_id": RUN_ID, "model": "AACNN_Adaptive", "model_sha256": sha256_file(selected_dir / "model.keras"),
              "class_order": CLASS_NAMES, "label": CLASS_NAMES[class_index], "scores": scores.tolist(),
              "classification_status": "PASS", "explanation": cam_status,
              "execution_source": "colab", "purpose": "uploaded_image_pipeline_check_not_dataset_test",
              "preprocessing_version": PREPROCESS_VERSION, "vit_revision": RESOLVED_VIT_REVISION,
              "classification_wall_seconds_including_vit_load": classification_seconds,
              "pipeline_wall_seconds": time.perf_counter()-start}
    write_json(dest / "result.json", result)
    original = rgb_image(image_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(original); axes[0].set_title("Ảnh đầu vào")
    axes[1].imshow(original)
    if cam_status.get("status") == "PASS":
        resized_cam = tf.image.resize(cam[...,None], [original.height, original.width]).numpy()[...,0]
        axes[1].imshow(resized_cam, cmap="jet", alpha=0.4, vmin=0, vmax=1)
    axes[1].set_title("Grad-CAM nhánh CNN — " + cam_status["status"])
    for ax in axes: ax.axis("off")
    fig.tight_layout(); fig.savefig(dest / "prediction_overlay.png", dpi=160); plt.show(); plt.close(fig)
    print("Nhãn dự đoán:", result["label"])
    for name, score in zip(CLASS_NAMES, scores): print(f"  {name}: {float(score):.6f}")
    print("Scores chưa được hiệu chuẩn thành xác suất mắc bệnh.")
    print("Đầu ra pipeline:", dest)
    print("Trạng thái Grad-CAM:", cam_status)
    return result, dest

upload_widget = widgets.FileUpload(accept=".png,.jpg,.jpeg", multiple=True,
                                  description="Chọn ảnh CT")
predict_button = widgets.Button(description="Dự đoán và tạo Grad-CAM", button_style="primary",
                                layout=widgets.Layout(width="260px"))
upload_output = widgets.Output()

def on_predict_clicked(_):
    """Chỉ suy luận khi nhấn nút; không chạy lại fit hoặc đổi tập đánh giá."""
    with upload_output:
        clear_output(wait=True)
        if not upload_widget.value:
            print("Chọn ít nhất một ảnh JPEG/PNG trước."); return
        predict_button.disabled = True
        try:
            entries = upload_widget.value
            # Hỗ trợ cả dạng dict của ipywidgets 7 và tuple của ipywidgets 8.
            items = entries.values() if isinstance(entries, dict) else entries
            for item in items:
                name = item.get("name", item.get("metadata", {}).get("name", "image.png"))
                try:
                    predict_uploaded_bytes(item["content"], name)
                except Exception as error:
                    print("Không xử lý được ảnh", name, ":", repr(error))
        finally:
            predict_button.disabled = False

predict_button.on_click(on_predict_clicked)
display(widgets.VBox([upload_widget, predict_button, upload_output]))
print("Chọn ảnh và nhấn nút. Model đã train được dùng để dự đoán, không train lại.")


## 20 Nhật ký kiểm tra kỹ thuật trước khi bàn giao notebook

Rà soát ngày 24/10/2026 trên TensorFlow CPU 2.18.1, không tải pretrained ViT và không dùng dataset nghiên cứu:

- Cả bốn builder: một bước cập nhật trọng số trên ảnh/feature tổng hợp; lưu/nạp `.keras` và so output trong dung sai — PASS.
- Adaptive: cập nhật thêm stage B với chính sách mở layer cuối và giữ BatchNorm frozen — PASS trên ảnh tổng hợp 32×32.
- Adaptive: forward đúng kích thước 512×512 cùng CLS 768; scores hữu hạn, tổng gần 1; gradient Grad-CAM liên thông và CAM 16×16 — PASS với ResNet khởi tạo ngẫu nhiên.
- Đã bổ sung `build()` cho AdaptiveGate để loại cảnh báo trạng thái chưa build khi serialization; kiểm tra lại save/load và stage B — PASS.
- Kiểm tra byte/pixel dedup và chặn trùng khác nhãn bằng dữ liệu tổng hợp — PASS.
- Cú pháp các cell, `custom_layers.py` và `predict.py` nhúng — PASS.

Những kiểm tra này không cung cấp accuracy thực, không chứng minh ViT pretrained trích đặc trưng đúng trên dataset, không kiểm chứng trọn Run All trên Colab, gói ZIP hoàn chỉnh, widget tải ảnh hoặc runtime điện thoại. Cần chạy thật, lưu outputs và đối chiếu trước khi nghiệm thu. Không biến nhật ký kỹ thuật này thành kết quả thí nghiệm của hai khóa luận.

Kiểm tra bổ sung: Keras Sequence fusion qua `fit` với class weights, callback macro F1 và lưu checkpoint — PASS trên dữ liệu tổng hợp. Chia 300 mẫu thành 210/45/45, không giao hash — PASS. Hàm nhận PNG, dựng inputs tổng hợp, inference/CAM và lưu tensor/JSON/overlay — PASS; bytes không phải ảnh bị từ chối. Tạo augmentation train-only 20 ảnh thành 40 mẫu, parent lineage và heldout separation — PASS trên dữ liệu tổng hợp. Những số lượng này là test kỹ thuật, không phải số mẫu hoặc metrics của dataset nghiên cứu.


## 21 Ứng viên nhẹ hơn giữ nguyên kiến trúc Adaptive

Chúng tôi giữ gói FP32 chính làm đối chứng. Bản ứng viên chỉ lưu trọng số **nhánh ViT** bằng FP16, không thay topology hoặc huấn luyện một kiến trúc khác. Khi chạy CPU, weights được nạp về float32 để tính; do đó đây chủ yếu là **giảm dung lượng lưu/truyền**, chưa phải bằng chứng giảm RAM hoặc tăng tốc trên điện thoại. Phần fusion Keras vẫn giữ nguyên. Dung lượng thực tế và sai lệch phải được đo, không gán trước tỷ lệ giảm.

Ứng viên được kiểm tra trên toàn validation: nhãn khớp với FP32, độ lệch scores và macro F1 so nhãn thật. Ngưỡng đề xuất trước khi chạy: giảm macro F1 không quá 0.01 và khớp nhãn ít nhất 98%. Đây là tiêu chí kỹ thuật của bản nén, không phải chuẩn lâm sàng. Không dùng test để quyết định giữ bản nén. Test trong run hiện tại đánh giá bản FP32 chính; **chưa có đánh giá test độc lập của ứng viên nhẹ**.

Chỉ khi qua kiểm tra validation mới tạo `LungCT_LightCandidate_<RUN_ID>.zip`. Gói này được ghi CANDIDATE; không thay thế gói chính hoặc tự coi mobile-ready. Nếu không qua, giữ nguyên FP32 và lưu lý do FAIL. Trước dùng trong app phải có runtime hỗ trợ, parity nhiều ảnh, kiểm thử Grad-CAM và số đo thiết bị. Quantization mobile sâu hơn hoặc FP16 fusion là bước chuyển đổi riêng, không được coi đã thực hiện ở đây.


In [ ]:
# Thực hiện sau gói FP32 chính; không tune bằng test.
light_dir = RUN_DIR / "light_candidate"
light_dir.mkdir(exist_ok=True)
quantized_vit_path = light_dir / "vit"
# Lưu half weights từ đúng ViT đã có trong gói, không tải revision khác.
vit_half = ViTModel.from_pretrained(bundle_dir / "vit", local_files_only=True).to("cpu").half().eval()
vit_half.save_pretrained(quantized_vit_path, safe_serialization=True)
processor.save_pretrained(quantized_vit_path)
del vit_half; gc.collect()
vit_light = ViTModel.from_pretrained(quantized_vit_path, local_files_only=True).to("cpu").float().eval()
light_processor = ViTImageProcessor.from_pretrained(quantized_vit_path, local_files_only=True)
light_features = []
for start in range(0, len(val_rows), VIT_BATCH_SIZE):
    rows = val_rows[start:start+VIT_BATCH_SIZE]
    images = [rgb_image(BASE_DIR / r["relative_path"]).resize(VIT_SIZE, Image.Resampling.NEAREST) for r in rows]
    batch = light_processor(images=images, return_tensors="pt")
    with torch.inference_mode():
        light_features.append(vit_light(**batch).last_hidden_state[:,0,:].numpy().astype(np.float32))
light_features = np.concatenate(light_features)
del vit_light; gc.collect()
_, fp32_scores = predict_sequence(selected_model, DataSequence(val_rows, features["validation"], "fusion"))
y_onehot, light_scores = predict_sequence(selected_model, DataSequence(val_rows, light_features, "fusion"))
y_true = y_onehot.argmax(1)
fp32_f1 = float(f1_score(y_true, fp32_scores.argmax(1), average="macro", zero_division=0))
light_f1 = float(f1_score(y_true, light_scores.argmax(1), average="macro", zero_division=0))
agreement = float(np.mean(fp32_scores.argmax(1) == light_scores.argmax(1)))
accepted = light_f1 >= fp32_f1 - LIGHT_MAX_MACRO_F1_DROP and agreement >= LIGHT_MIN_LABEL_AGREEMENT
fp32_vit_bytes = sum(f.stat().st_size for f in (bundle_dir / "vit").rglob("*") if f.is_file())
light_vit_bytes = sum(f.stat().st_size for f in quantized_vit_path.rglob("*") if f.is_file())
light_report = {"status": "PASS_VALIDATION_ONLY" if accepted else "FAIL_VALIDATION", "topology_changed": False,
    "change": "ViT FP16 weight storage, CPU float32 compute; fusion unchanged", "split": "validation",
    "n_images": len(val_rows), "fp32_macro_f1": fp32_f1, "candidate_macro_f1": light_f1,
    "label_agreement": agreement, "max_abs_score_error": float(np.max(np.abs(fp32_scores-light_scores))),
    "fp32_vit_bytes": fp32_vit_bytes, "candidate_vit_bytes": light_vit_bytes,
    "thresholds": {"max_macro_f1_drop": LIGHT_MAX_MACRO_F1_DROP, "min_label_agreement": LIGHT_MIN_LABEL_AGREEMENT},
    "mobile_ready": False, "candidate_independent_test": "NOT_RUN", "mobile_ram_latency": "NOT_MEASURED"}
write_json(light_dir / "light_validation_report.json", light_report)
print(light_report)
if accepted:
    for path in bundle_dir.iterdir():
        if path.name in {"vit", "bundle_manifest.json", "offline_parity.json", "offline_prediction.json"}: continue
        target = light_dir / path.name
        if path.name == "fusion":
            target.mkdir(exist_ok=True)
            for model_file in ("model.keras", "best.weights.h5"):
                shutil.copy2(path / model_file, target / model_file)
            shutil.copy2(path / "artifact_manifest.json", light_dir / "fp32_reference_manifest.json")
        elif path.is_dir(): shutil.copytree(path, target, dirs_exist_ok=True)
        else: shutil.copy2(path, target)
    (light_dir / "CANH_BAO_UNG_VIEN.txt").write_text("Đây là ứng viên ViT lưu FP16, fusion không đổi. Chỉ có kiểm tra validation của ứng viên. Không sao chép test metrics FP32 thành kết quả của ứng viên. fp32_reference_manifest.json chỉ là nguồn tham chiếu. Chưa mobile-ready, chưa đo RAM/latency điện thoại. Không có offline_parity của ứng viên. Đọc light_validation_report.json trước dùng.", encoding="utf-8")
    # Không mang kết quả parity FP32 sang ứng viên như bằng chứng đã kiểm tra ứng viên.
    write_json(light_dir / "bundle_manifest.json", {"primary_reference_run": RUN_ID, "candidate": True,
        "mobile_ready": False, "change": light_report["change"], "validation_report": "light_validation_report.json",
        "files": {str(f.relative_to(light_dir)): {"sha256": sha256_file(f), "bytes": f.stat().st_size}
                  for f in sorted(light_dir.rglob("*")) if f.is_file() and f.name != "bundle_manifest.json"}})
    candidate_zip = RUN_DIR / f"LungCT_LightCandidate_{RUN_ID}.zip"
    with zipfile.ZipFile(candidate_zip, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(light_dir.rglob("*")):
            if path.is_file(): archive.write(path, str(path.relative_to(light_dir)))
    write_json(RUN_DIR / "light_package_summary.json", {"path": str(candidate_zip),
        "sha256": sha256_file(candidate_zip), "bytes": candidate_zip.stat().st_size,
        "fp32_package_bytes": zip_path.stat().st_size, "status": "CANDIDATE_VALIDATION_ONLY"})
    print("Ứng viên nhẹ để đánh giá tiếp, CHƯA mobile-ready:", candidate_zip)
else:
    print("Không tạo ZIP ứng viên đạt; giữ gói FP32 chính và xem light_validation_report.json.")


## 22 Ghi nhận thay đổi về tổng quát hóa và dung lượng

1. Giữ kiến trúc Adaptive theo lựa chọn của người thực hiện; không tạo mô hình student hoặc tự thay backbone.
2. Giữ 70/15/15 trên ảnh gốc đã làm sạch; chỉ tạo biến thể từ train. CNN và ViT lấy cùng ảnh augmentation để tránh lệch hai nhánh. Ghi manifest parent hash và số lượng trước/sau.
3. Tắt augmentation online đơn nhánh ở mặc định; giữ Dropout/L2/AdamW, early stopping theo validation macro F1 và chính sách BatchNorm đã ghi. Các điều chỉnh này cần đo, không được tự mô tả là đã tăng generalization.
4. Thêm nhánh xuất ứng viên lưu ViT FP16 sau train, giữ full FP32 làm đối chứng. Đo dung lượng thực và sai lệch/F1 trên validation; không tự nhận tăng tốc hoặc giảm RAM điện thoại.
5. Không thay gói mô hình chính bằng candidate chỉ vì file nhỏ hơn. Cần đánh giá ngoài dữ liệu huấn luyện, kiểm thử chuyển đổi và Grad-CAM trên thiết bị trước bàn giao app.
6. Pipeline upload phục vụ xem dự đoán và tensor từng bước; ảnh ngoài không có nhãn không chứng minh độ chính xác. Tập external có nhãn và protocol phải được xây riêng và không dùng để tune sau khi gọi là external test.

**Trạng thái mới bổ sung:** augmentation đồng bộ và xuất FP16 ViT đã có code nhưng chưa chạy end-to-end với pretrained/dataset thật trong lần biên soạn này. Phải kiểm tra outputs khi chạy. Thí nghiệm kỹ thuật tổng hợp ở mục 20 không được dùng thay bằng chứng của các bước mới.


## 23 Sửa lỗi lệch số ảnh và feature sau augmentation

Phiên bản 1.5 sửa lỗi tích hợp của bản 1.4: train_rows đã mở rộng bằng augmentation nhưng feature train vẫn lấy từ splits["train"] gốc. Điều đó làm số ảnh lớn hơn số CLS và dừng ở DataSequence. Đã đổi sang FEATURE_ROWS lấy đúng train_rows sau augmentation, kiểm tra số lượng và thứ tự/hash trước preflight. Cell augmentation lấy ảnh gốc từ split đã khóa để chạy lại không nhân đôi biến thể. Không bỏ assert hoặc cắt mảng nhằm giấu lỗi; giữ tỷ lệ split gốc 70/15/15.


## Dấu vết tệp đầu vào

- `demo_test_combined.py` SHA256: `341a9b140238a9fd47df92c5854b21345af15dedc2b049d082bb28afee3a0824`
- `Demo_Test_Combined.ipynb` SHA256: `710d80caa220dddc6d65a9a592a8ca7911211bb138a13a3273d9323e6350015d`